In [ ]:
/**
 * Database Architecture & Design:
 * > Databases are the heart of the system. They determine scalability, reliability and
 *   performance
 * > Database Fundamentals:
 *   - What is a Database really do?
 *   - Why not just use files?
 *   - Relational vs Non-Relational Databases
 *   - Choosing the Right Database
 *   - The paradigm shift toward real-time databases
 * 
 * Note: There is nobody in the world who can tell us what is the right
 *       database to use.
*/

In [ ]:
/**
 * ============================================================
 * Database Architecture & Design
 * ============================================================
 *
 *   Databases are the heart of any system. They determine
 *   scalability, reliability, and performance.
 *
 *   Note: There is no one in the world who can tell you what
 *         the "right" database to use is. It always depends
 *         on your use case, traffic, and data structure.
 *
 *
 * ───────────────────────────────────────────────────────
 * CONCEPT 1 — What is a Database, Really?
 * ───────────────────────────────────────────────────────
 *
 *   At its core, a database is just a disk.
 *   All your data ultimately lives on a disk — whether you
 *   use a database or not.
 *
 *   So the real question is: why not just use plain files?
 *
 *
 *   The flat file approach (JSON files):
 *
 *    users.json
 *    ┌──────────────────────────────────────┐
 *    │  [                                   │
 *    │    { "id": 1, "name": "Hitesh" },    │
 *    │    { "id": 2, "name": "Piyush" },    │
 *    │    { "id": 3, "name": "Aman"   }     │
 *    │  ]                                   │
 *    └──────────────────────────────────────┘
 *
 *    To read user with id = 2:
 *    1. Read the entire file into memory
 *    2. Parse the JSON
 *    3. Loop through every record to find id = 2
 *    4. Return it
 *
 *    This works perfectly fine for a single user. ✓
 *
 *
 *   The problem — what happens at scale:
 *
 *    ┌──────────────────────────────────────────────────┐
 *    │  users.json = 10 million records = 2 GB file     │
 *    │                                                  │
 *    │  Request 1: Read user id = 5                     │
 *    │  → load 2 GB into memory                         │
 *    │  → parse entire JSON                             │
 *    │  → loop through 10M records                      │
 *    │  → return 1 record                               │
 *    │  → time taken: several seconds ✗✗               │
 *    │                                                  │
 *    │  Request 2 (arrives 10ms later): Update user id=5│
 *    │  → Request 1 is still reading the file           │
 *    │  → Both try to write at the same time            │
 *    │  → File gets corrupted ✗✗                       │
 *    └──────────────────────────────────────────────────┘
 *
 *
 * ───────────────────────────────────────────────────────
 * CONCEPT 2 — What Does a Database Actually Give You?
 * ───────────────────────────────────────────────────────
 *
 *   A database is not magic. It is a layer between your
 *   application and the disk that provides a set of
 *   guarantees that plain files cannot.
 *
 *   Here are the six guarantees a database provides:
 *
 *
 *   Guarantee 1 — Concurrency Management:
 *
 *   - What it is: If two processes try to read and write
 *     the same data at the same time, the database manages
 *     who can write and who has to wait.
 *   - With flat files: both processes write simultaneously
 *     → file gets corrupted → data is lost.
 *   - With a database: locking and concurrency control
 *     ensures writes happen one at a time safely.
 *
 *    VISUAL — Concurrency problem with flat files:
 *
 *    Process A                       Process B
 *    reads users.json                reads users.json
 *         │                               │
 *         │  modifies record              │  modifies record
 *         │                               │
 *         ▼                               ▼
 *    writes file ──────────────────────── writes file
 *    (overwrites B's changes ✗)     (overwrites A's changes ✗)
 *    → one update is silently lost
 *
 *    VISUAL — Concurrency with a Database:
 *
 *    Process A                       Process B
 *    requests write lock              requests write lock
 *         │                               │
 *         ▼                               │  waits...
 *    gets lock ✓                         │
 *    writes safely                        │
 *    releases lock                        │
 *         │                               ▼
 *         │                          gets lock ✓
 *         │                          writes safely
 *    Both updates succeed ✓ in order
 *
 *
 *   Guarantee 2 — Smart Querying:
 *
 *   - What it is: A database lets you fetch only the
 *     records you need without reading everything.
 *   - With flat files: you must read the entire file,
 *     load it into memory, and loop through every record.
 *   - With a database: use a query to fetch exactly
 *     what you need using indexes — extremely fast.
 *
 *    Flat file (slow):
 *    Read 2 GB → parse → loop 10M records → return 1 ✗
 *
 *    Database with index (fast):
 *    SELECT * FROM users WHERE id = 5
 *    → index lookup → jump directly to record → return 1 ✓
 *    (milliseconds, regardless of how many records exist)
 *
 *
 *   Guarantee 3 — Data Integrity:
 *
 *   - What it is: The database enforces rules about what
 *     kind of data can be stored.
 *   - With flat files: anyone can write anything.
 *     age = "banana" → nobody stops it → silent corruption.
 *   - With a database: you define a schema.
 *     age must be an INTEGER → the database rejects "banana".
 *
 *    Flat file — no protection:
 *    ┌──────────────────────────────────────┐
 *    │  { "name": "Hitesh", "age": "banana"}│  ← allowed ✗
 *    └──────────────────────────────────────┘
 *
 *    Database — schema enforced:
 *    ┌──────────────────────────────────────┐
 *    │  age column: INTEGER NOT NULL        │
 *    │  INSERT age = "banana"               │
 *    │  → ERROR: invalid input for integer  │  ← rejected ✓
 *    └──────────────────────────────────────┘
 *
 *
 *   Guarantee 4 — Crash Recovery (Most Important):
 *
 *   - What it is: If the server crashes mid-write, the
 *     database can recover to a consistent state.
 *     No data corruption, no partial writes stuck forever.
 *   - This is the most critical feature a database provides.
 *   - It is also the foundation that enables all other features.
 *   - How it works: databases use a Write-Ahead Log (WAL).
 *     Before writing data to disk, they first write what
 *     they are about to do to a log file.
 *     On crash recovery, the database reads the log and
 *     replays or reverses incomplete operations.
 *
 *    VISUAL — Crash mid-write:
 *
 *    Without database (flat file):
 *    Writing 10,000 records... server crashes at record 5,000
 *    → File has 5,000 records written, 5,000 missing
 *    → File is half-written and corrupt ✗
 *    → No way to know what was written and what was not ✗
 *
 *    With database (WAL):
 *    Step 1: Write to WAL log: "about to insert 10,000 records"
 *    Step 2: Write actual data...
 *    Server crashes at record 5,000
 *         │
 *         ▼
 *    Database restarts
 *    Reads WAL log: "insert was incomplete"
 *    Either completes the insert OR rolls it back entirely
 *    → Database is always in a consistent state ✓
 *
 *
 *   Guarantee 5 — Partial Update Control (Transactions):
 *
 *   - What it is: A transaction groups multiple operations
 *     into one unit. Either ALL of them succeed, or NONE of
 *     them happen. There is no in-between.
 *   - This is critical for operations that must be atomic.
 *
 *    Classic example — bank transfer:
 *
 *    Transfer ₹1000 from Hitesh to Piyush:
 *    Step 1: Deduct ₹1000 from Hitesh's account
 *    Step 2: Add ₹1000 to Piyush's account
 *
 *    Without transactions:
 *    Step 1 succeeds → server crashes → Step 2 never runs
 *    → ₹1000 deducted from Hitesh but never added to Piyush
 *    → Money disappears ✗
 *
 *    With transactions:
 *    BEGIN TRANSACTION
 *      Step 1: Deduct ₹1000 from Hitesh   ✓
 *      Step 2: Add ₹1000 to Piyush        ✓
 *    COMMIT  ← only saved if BOTH succeed
 *
 *    If Step 2 fails → entire transaction ROLLED BACK
 *    → Hitesh's money is restored ✓ → consistent state ✓
 *
 *    VISUAL — Transaction as an all-or-nothing box:
 *
 *    ┌──────────────────────────────────────────┐
 *    │  TRANSACTION                              │
 *    │                                          │
 *    │  Step 1: Deduct ₹1000 from Hitesh        │
 *    │  Step 2: Add ₹1000 to Piyush             │
 *    │                                          │
 *    │  Both succeed → COMMIT → saved ✓         │
 *    │  Any step fails → ROLLBACK → nothing ✓   │
 *    └──────────────────────────────────────────┘
 *
 *
 *   Guarantee 6 — Efficient Storage on Disk:
 *
 *   - Databases store data on disk in optimised formats
 *     (B-trees, LSM trees) that make reads and writes fast.
 *   - They manage memory caching (buffer pool) — frequently
 *     accessed data stays in RAM, rarely accessed data stays
 *     on disk.
 *   - Plain files have none of this — every read hits the
 *     disk directly.
 *
 *
 *   All six guarantees together — summary:
 *
 *     +---------------------+--------------------+--------------------+
 *     |  Guarantee          |  Flat File         |  Database          |
 *     +---------------------+--------------------+--------------------+
 *     |  Concurrency        |  Corruption risk   |  Managed safely    |
 *     |  Smart Querying     |  Read entire file  |  Index lookup      |
 *     |  Data Integrity     |  Anything allowed  |  Schema enforced   |
 *     |  Crash Recovery     |  Data lost / corrupt| WAL + recovery    |
 *     |  Transactions       |  No control        |  All or nothing    |
 *     |  Efficient Storage  |  Raw disk access   |  Optimised formats |
 *     +---------------------+--------------------+--------------------+
 *
 *
 * ───────────────────────────────────────────────────────
 * CONCEPT 3 — Crash Recovery is the Foundation
 * ───────────────────────────────────────────────────────
 *
 *   Crash recovery is not just one feature — it is the
 *   foundation that makes everything else possible.
 *
 *   Think about it:
 *   - Transactions work because crash recovery ensures
 *     a partial transaction can be rolled back on restart.
 *   - Data integrity works because crash recovery ensures
 *     a half-written schema change does not corrupt the table.
 *   - Concurrency works because crash recovery ensures
 *     locks are properly released even after a crash.
 *
 *   Without crash recovery → none of the other guarantees
 *   can be trusted → you don't have a real database.
 *
 *
 *   VISUAL — Crash recovery is the base layer:
 *
 *    ┌──────────────────────────────────────────────────┐
 *    │             Transactions                         │
 *    ├──────────────────────────────────────────────────┤
 *    │             Data Integrity                       │
 *    ├──────────────────────────────────────────────────┤
 *    │             Concurrency Control                  │
 *    ├──────────────────────────────────────────────────┤
 *    │             Smart Querying                       │
 *    ├──────────────────────────────────────────────────┤
 *    │    CRASH RECOVERY  ← foundation of everything    │
 *    └──────────────────────────────────────────────────┘
 *    │             Disk (Physical Storage)              │
 *    └──────────────────────────────────────────────────┘
 *
 *   This is why even the simplest production system should
 *   never store critical data in plain files.
 *   The moment you need reliability → you need a database.
 */


In [ ]:
/**
 * ============================================================
 * Core Components of a Database
 * ============================================================
 *
 *   A database is not one single thing — it is made up of
 *   several internal components that work together to give
 *   you all the guarantees we discussed (concurrency, queries,
 *   crash recovery, transactions etc.).
 *
 *   Understanding these components helps you understand WHY
 *   databases behave the way they do, and why certain
 *   design decisions (like indexing) matter so much.
 *
 *
 *   VISUAL — All components inside a database:
 *
 *    ┌─────────────────────────────────────────────────────────────┐
 *    │                     DATABASE                                │
 *    │                                                             │
 *    │  ┌──────────────┐    ┌──────────────────────────────────┐   │
 *    │  │    Client    │    │         Query Layer              │   │
 *    │  │  (your app)  │───►│  Query Processor + Optimizer     │   │
 *    │  └──────────────┘    └────────────────┬─────────────────┘   │
 *    │                                       │                     │
 *    │                       ┌───────────────┼───────────────┐     │
 *    │                       │               │               │     │
 *    │                       ▼               ▼               ▼     │
 *    │              ┌──────────────┐ ┌──────────────┐ ┌────────┐   │
 *    │              │ Transaction  │ │    Lock      │ │ Buffer │   │
 *    │              │   Manager    │ │   Manager    │ │  Pool  │   │
 *    │              └──────┬───────┘ └──────────────┘ └───┬────┘   │
 *    │                     │                              │        │
 *    │                     ▼                              ▼        │
 *    │              ┌─────────────────────────────────────────┐    │
 *    │              │           Storage Engine                │    │
 *    │              │         Pages → Blocks → Disk           │    │
 *    │              └─────────────────┬───────────────────────┘    │
 *    │                                │                            │
 *    │              ┌─────────────────┼───────────────────────┐    │
 *    │              │                 │                       │    │
 *    │              ▼                 ▼                       ▼    │
 *    │           ┌──────┐         ┌───────┐            ┌─────────┐ │
 *    │           │ WAL  │         │  Data │            │Metadata │ │
 *    │           │ Log  │         │ Files │            │  Store  │ │
 *    │           └──────┘         └───────┘            └─────────┘ │
 *    │                                                             │
 *    │                       Physical Disk                         │
 *    └─────────────────────────────────────────────────────────────┘
 *
 *
 * ───────────────────────────────────────────────────────
 * COMPONENT 1 — Storage Engine
 * ───────────────────────────────────────────────────────
 *
 *   - The Storage Engine is the most important component
 *     of any database. Everything else depends on it.
 *   - It governs how data is stored on disk and how it
 *     is retrieved back into memory.
 *   - It decides the internal format: pages, blocks,
 *     tree structures, and how they map to physical disk.
 *
 *
 *   How data actually lives on a disk:
 *
 *   - A disk does not work at the level of individual bytes
 *     or individual records.
 *   - A disk works in sectors and blocks.
 *   - When you ask the OS to read one byte, the OS does not
 *     fetch just that one byte — it fetches the entire block
 *     (sector) that contains that byte.
 *   - Block size is typically 512 bytes to 4 KB depending
 *     on the OS and disk type.
 *
 *
 *   VISUAL — You ask for 1 record, disk gives you a block:
 *
 *    Your query: SELECT * FROM users WHERE id = 5
 *    You expect: just the row for id = 5
 *
 *    What actually happens on disk:
 *
 *    ┌──────────────────────────────────────────────────────┐
 *    │                  DISK BLOCK (8 KB)                   │
 *    │                                                      │
 *    │  id=1 | id=2 | id=3 | id=4 | id=5 | id=6 | id=7 ...  │
 *    │                          ↑                           │
 *    │                     what you wanted                  │
 *    │                                                      │
 *    │  But disk gives you the ENTIRE block — all records   │
 *    │  in that block are loaded into memory together       │
 *    └──────────────────────────────────────────────────────┘
 *
 *    This is why no matter how well you optimise your queries
 *    or how many indexes you add — you can never fetch just
 *    one record. You always get the entire block. The database
 *    then finds your specific record inside that block.
 *
 *
 *   Pages — how the Storage Engine organises blocks:
 *
 *   - The OS works in blocks. The database works in pages.
 *   - A page is the database's own unit of storage —
 *     typically 8 KB or 16 KB (configurable).
 *   - One page maps to one or more OS blocks on disk.
 *   - The storage engine reads and writes data page by page,
 *     not record by record.
 *   - When data is written to disk, it is stored in page
 *     format, which the OS then replicates as blocks on the
 *     actual hard disk. This is how every database works.
 *
 *
 *   VISUAL — Pages vs Blocks vs Records:
 *
 *    Your Application
 *         │
 *         │  "give me user id = 5"
 *         ▼
 *    ┌──────────────────────────────────────┐
 *    │  Database Page (8 KB)                │
 *    │  ┌─────────┬─────────┬─────────┐     │
 *    │  │ Record 1│ Record 2│ Record 3│     │
 *    │  │ id=4    │ id=5    │ id=6    │     │
 *    │  └─────────┴─────────┴─────────┘     │
 *    │  Header | Free Space | Slot Array    │
 *    └──────────────────────────────────────┘
 *         │
 *         │  maps to
 *         ▼
 *    ┌──────────────────────────────────────┐
 *    │  OS Blocks (512 bytes each)          │
 *    │  Block 1 | Block 2 | ... | Block 16  │
 *    │  (one 8KB page = 16 × 512B blocks)   │
 *    └──────────────────────────────────────┘
 *         │
 *         │  stored on
 *         ▼
 *    Physical Hard Disk / SSD
 *
 *    Rule: no matter what you do, the minimum unit of
 *    I/O is one page. You cannot read less than one page.
 *
 *
 *   Why the Storage Engine is the most important component:
 *   - All performance, all durability, all crash recovery
 *     starts here.
 *   - Popular storage engines:
 *     InnoDB (MySQL default), RocksDB (used by many NoSQL),
 *     WiredTiger (MongoDB default), B-tree vs LSM tree based.
 *   - Different storage engines have very different
 *     performance characteristics for reads vs writes.
 *
 *
 * ───────────────────────────────────────────────────────
 * COMPONENT 2 — Query Processor
 * ───────────────────────────────────────────────────────
 *
 *   - The Query Processor receives the SQL query from your
 *     application, parses it, and prepares an execution plan.
 *   - It translates the human-readable SQL into a series of
 *     internal operations the database can execute.
 *
 *
 *   What Query Processor does step by step:
 *
 *    Your query:
 *    SELECT name FROM users
 *    WHERE age > 25
 *    GROUP BY city
 *    ORDER BY name
 *
 *         │
 *         ▼
 *    Step 1: PARSING
 *    Checks syntax — is this valid SQL?
 *    Breaks it into a parse tree (internal structure)
 *         │
 *         ▼
 *    Step 2: SEMANTIC ANALYSIS
 *    Does the table "users" exist?
 *    Does column "age" exist in users?
 *    Is the query logically valid?
 *         │
 *         ▼
 *    Step 3: EXECUTION PLAN
 *    Builds a logical plan of operations:
 *    Scan users → Filter age > 25 → Group by city → Sort by name
 *         │
 *         ▼
 *    Passes plan to Query Optimizer
 *
 *
 *   Important — the Query Processor ignores the order you wrote:
 *
 *   - You might write WHERE before GROUP BY thinking that is
 *     how it will execute. It is good practice — but the
 *     Query Processor does not care about the order you wrote.
 *   - It builds its own logical execution plan based on what
 *     will be most efficient — not what you wrote.
 *   - Humans are not always the best at writing optimal queries,
 *     so the software was made smarter.
 *   - This is also why SQL is called a declarative language:
 *     you declare WHAT you want, not HOW to get it.
 *
 *
 *   VISUAL — Your order vs execution order:
 *
 *    What you wrote:            What actually executes:
 *    ┌──────────────┐           ┌──────────────────────────┐
 *    │ SELECT       │           │ 1. FROM users            │
 *    │ FROM users   │           │    (identify the table)  │
 *    │ WHERE age>25 │   ─────►  │ 2. WHERE age > 25        │
 *    │ GROUP BY city│           │    (filter rows first)   │
 *    │ ORDER BY name│           │ 3. GROUP BY city         │
 *    └──────────────┘           │ 4. SELECT name           │
 *                               │ 5. ORDER BY name         │
 *                               └──────────────────────────┘
 *    Query Processor reorders for optimal execution ✓
 *
 *
 * ───────────────────────────────────────────────────────
 * COMPONENT 3 — Query Optimizer
 * ───────────────────────────────────────────────────────
 *
 *   - The Query Optimizer works alongside the Query Processor.
 *   - Its one and only job is speed — how fast can this query run?
 *   - It takes the logical plan from the Query Processor and
 *     finds the most efficient way to execute it.
 *   - It decides: which index to use, in what order to join
 *     tables, whether to scan the full table or use an index.
 *
 *
 *   How the Query Optimizer makes decisions:
 *
 *    Query: SELECT * FROM orders WHERE user_id = 5
 *
 *    Option A: Full table scan
 *    → Read every single row in orders table
 *    → Check if user_id = 5
 *    → Return matching rows
 *    → Cost: slow if table has millions of rows ✗
 *
 *    Option B: Index on user_id exists
 *    → Jump directly to index entry for user_id = 5
 *    → Fetch only those rows
 *    → Cost: milliseconds regardless of table size ✓
 *
 *    Optimizer picks Option B automatically ✓
 *
 *
 *   VISUAL — Query Optimizer choosing execution path:
 *
 *    ┌──────────────────────────────────────────────────┐
 *    │  Query Optimizer                                 │
 *    │                                                  │
 *    │  Query: SELECT * FROM orders WHERE user_id = 5   │
 *    │                                                  │
 *    │  Checks statistics:                              │
 *    │  - orders table has 10 million rows              │
 *    │  - index exists on user_id column                │
 *    │  - estimated matching rows: ~50                  │
 *    │                                                  │
 *    │  Decision: use index on user_id         ✓        │
 *    │  Estimated cost: 2ms                             │
 *    │  vs full scan cost: 8 seconds           ✗        │
 *    └──────────────────────────────────────────────────┘
 *
 *    The optimizer uses statistics it collects about
 *    your data (row counts, value distributions) to
 *    estimate costs and pick the fastest plan.
 *
 *
 * ───────────────────────────────────────────────────────
 * COMPONENT 4 — Transaction Manager
 * ───────────────────────────────────────────────────────
 *
 *   - The Transaction Manager handles ACID properties,
 *     lock management, and isolation levels.
 *   - It ensures that when you run a transaction, all the
 *     ACID guarantees are maintained.
 *
 *   ACID stands for:
 *
 *     +------------------+--------------------------------------------+
 *     |  Property        |  What it means                             |
 *     +------------------+--------------------------------------------+
 *     |  Atomicity       |  All operations in a transaction succeed   |
 *     |                  |  or none of them happen. No half-updates.  |
 *     +------------------+--------------------------------------------+
 *     |  Consistency     |  A transaction brings the database from    |
 *     |                  |  one valid state to another valid state.   |
 *     |                  |  All rules (constraints, foreign keys)     |
 *     |                  |  must be satisfied before and after.       |
 *     +------------------+--------------------------------------------+
 *     |  Isolation       |  Concurrent transactions do not interfere  |
 *     |                  |  with each other. Each transaction sees    |
 *     |                  |  a consistent snapshot of the data.        |
 *     +------------------+--------------------------------------------+
 *     |  Durability      |  Once a transaction is committed, it is    |
 *     |                  |  permanent — even if the server crashes    |
 *     |                  |  immediately after. (This is the WAL.)     |
 *     +------------------+--------------------------------------------+
 *
 *
 * ───────────────────────────────────────────────────────
 * COMPONENT 5 — Lock Manager
 * ───────────────────────────────────────────────────────
 *
 *   - In older databases, the Transaction Manager handled
 *     both ACID and locking together.
 *   - Modern databases have separated these into two
 *     dedicated components for better performance.
 *   - The Lock Manager dedicatedly and concurrently manages
 *     all locking — who has a lock, who is waiting for a
 *     lock, and when to release locks.
 *   - This separation is why modern databases can handle
 *     thousands of concurrent transactions much faster
 *     than older databases could.
 *
 *
 *   VISUAL — Lock Manager controlling concurrent writes:
 *
 *    Transaction A            Transaction B
 *    wants to update          wants to update
 *    user id = 5              user id = 5
 *         │                        │
 *         ▼                        │
 *    [Lock Manager]                │
 *    grants WRITE lock             │
 *    to Transaction A              │  waits in queue
 *         │                        │
 *    A reads, modifies,            │
 *    writes user id = 5            │
 *    releases lock                 │
 *         │                        ▼
 *         │                   [Lock Manager]
 *         │                   grants lock to B
 *         │                   B reads updated value
 *         │                   modifies, writes, releases
 *
 *    Both transactions complete correctly ✓
 *    No corruption, no lost update ✓
 *
 *
 * ───────────────────────────────────────────────────────
 * COMPONENT 6 — Buffer Pool
 * ───────────────────────────────────────────────────────
 *
 *   - The Buffer Pool is the database's page caching system.
 *   - Reading from disk on every query is extremely slow.
 *   - The Buffer Pool keeps frequently accessed pages in
 *     RAM so the database does not have to go to disk again.
 *   - Think of it as the database's own RAM cache.
 *
 *
 *   VISUAL — Buffer Pool as a cache:
 *
 *    First time: SELECT * FROM users WHERE id = 5
 *         │
 *         │  page not in buffer pool
 *         ▼
 *    [Disk read — slow, ~10ms]
 *    Loads page into Buffer Pool
 *         │
 *         ▼
 *    Returns result to query
 *
 *    Second time: SELECT * FROM users WHERE id = 7
 *    (id=7 is in the same page as id=5)
 *         │
 *         │  page already in buffer pool!
 *         ▼
 *    [RAM read — fast, ~0.1ms]
 *    No disk access needed ✓
 *    Returns result instantly
 *
 *
 *    ┌──────────────────────────────────────────────────┐
 *    │             Buffer Pool (RAM)                    │
 *    │  ┌──────────┐ ┌──────────┐ ┌──────────┐          │
 *    │  │  Page 1  │ │  Page 7  │ │  Page 23 │ ...      │
 *    │  │ (hot)    │ │ (hot)    │ │ (cold)   │          │
 *    │  └──────────┘ └──────────┘ └──────────┘          │
 *    │  When buffer pool is full → evict coldest page   │
 *    │  (LRU — Least Recently Used algorithm)           │
 *    └──────────────────────────────────────────────────┘
 *         │
 *         │  miss (page not in pool)
 *         ▼
 *    ┌──────────────────────────────────────────────────┐
 *    │                   Disk                           │
 *    │           Data files with all pages              │
 *    └──────────────────────────────────────────────────┘
 *
 *    The Buffer Pool is why:
 *    - The first query is slow (cold start, hits disk)
 *    - Repeated queries are fast (data is in RAM)
 *    - Giving your database more RAM = better performance
 *
 *
 * ───────────────────────────────────────────────────────
 * COMPONENT 7 — WAL (Write-Ahead Log)
 * ───────────────────────────────────────────────────────
 *
 *   - WAL stands for Write-Ahead Log.
 *   - The rule is simple: before writing any change to a
 *     data page, first write that change to the log.
 *   - Only after the log entry is safely written does the
 *     database make the actual change to the data page.
 *   - "Write-Ahead" means the log is always written first —
 *     ahead of the actual data change.
 *
 *
 *   Why WAL enables Crash Recovery:
 *
 *   - When you commit a transaction, the database promises
 *     you: "your data is saved."
 *   - To make that promise safely, the change must be in the
 *     WAL log before saying "committed."
 *   - If the server crashes before the data page is updated,
 *     the WAL log has enough information to replay the change
 *     when the database restarts.
 *   - If nothing was written to the log, the database never
 *     made any promise — so there is nothing to recover.
 *
 *
 *   VISUAL — WAL write sequence:
 *
 *    UPDATE users SET age = 26 WHERE id = 5
 *         │
 *         ▼
 *    Step 1: Write to WAL log first
 *    ┌─────────────────────────────────────────────────┐
 *    │  WAL Log                                        │
 *    │  [LSN 1001] UPDATE users id=5 age: 25 → 26      │
 *    │  ← written to disk (durable)                    │
 *    └─────────────────────────────────────────────────┘
 *         │
 *         ▼
 *    Step 2: Update the data page in Buffer Pool (RAM)
 *    (not yet on disk — just in memory)
 *         │
 *         ▼
 *    Step 3: Return "committed" to the application ✓
 *
 *    Later (async):
 *    Step 4: Buffer Pool flushes updated page to disk
 *    (called a checkpoint)
 *
 *
 *   VISUAL — Crash scenario and recovery:
 *
 *    Server crashes after Step 2 but before Step 4
 *    (updated page is in RAM but not yet on disk)
 *         │
 *         ▼
 *    Server restarts
 *    Database reads WAL log:
 *    "LSN 1001: UPDATE users id=5 age: 25 → 26 — was committed"
 *         │
 *         ▼
 *    Replays the change on the data page
 *    Data is restored exactly as committed ✓
 *
 *    Server crashes before Step 1 (log not written):
 *    Database restarts
 *    No log entry found for this update
 *    Transaction is treated as never happened
 *    Rolled back silently ✓  (Atomicity maintained)
 *
 *
 * ───────────────────────────────────────────────────────
 * COMPONENT 8 — Metadata Store
 * ───────────────────────────────────────────────────────
 *
 *   - The Metadata Store (also called the System Catalog
 *     or Data Dictionary) stores information about the
 *     database itself — not the actual user data.
 *   - It is the database's own internal database.
 *
 *   What the Metadata Store contains:
 *
 *     +------------------+------------------------------------------+
 *     |  Metadata type   |  Example                                 |
 *     +------------------+------------------------------------------+
 *     |  Table schemas   |  users table has columns: id, name, age  |
 *     |  Column types    |  age is INTEGER, name is VARCHAR(100)    |
 *     |  Indexes         |  index on users.email exists             |
 *     |  Constraints     |  email must be UNIQUE, age NOT NULL      |
 *     |  User permissions|  user hitesh can SELECT on users table   |
 *     |  Statistics      |  users table has 10M rows, avg age = 28  |
 *     |  Foreign keys    |  orders.user_id references users.id      |
 *     +------------------+------------------------------------------+
 *
 *   - The Query Processor uses the Metadata Store to check
 *     if tables and columns exist.
 *   - The Query Optimizer uses it to read statistics for
 *     choosing the best execution plan.
 *   - The Transaction Manager uses it to enforce constraints.
 *
 *
 * ───────────────────────────────────────────────────────
 * COMPLETE PICTURE — How All Components Work Together
 * ───────────────────────────────────────────────────────
 *
 *   Query: UPDATE orders SET status = 'paid' WHERE id = 101
 *
 *    [Your Application]
 *         │  sends query
 *         ▼
 *    [Query Processor]
 *    Parses SQL, checks Metadata Store (does orders table exist?)
 *         │
 *         ▼
 *    [Query Optimizer]
 *    Checks index on orders.id → use index, skip full scan
 *         │
 *         ▼
 *    [Transaction Manager]
 *    Begins transaction, enforces ACID rules
 *         │
 *         ▼
 *    [Lock Manager]
 *    Acquires WRITE lock on row id = 101
 *         │
 *         ▼
 *    [Buffer Pool]
 *    Is the page for id = 101 already in RAM?
 *    YES → use it | NO → load from disk into pool
 *         │
 *         ▼
 *    [WAL — Write-Ahead Log]
 *    Write log entry: "UPDATE orders id=101 status: pending → paid"
 *    Flush log to disk (durable)
 *         │
 *         ▼
 *    [Storage Engine]
 *    Update the page in Buffer Pool
 *         │
 *         ▼
 *    [Transaction Manager]
 *    COMMIT — transaction is done
 *         │
 *         ▼
 *    [Lock Manager]
 *    Release WRITE lock on row id = 101
 *         │
 *         ▼
 *    [Your Application]
 *    Receives: UPDATE successful ✓
 *
 *    Later (async):
 *    [Buffer Pool] flushes updated page to disk (checkpoint)
 *
 *
 *   Component summary:
 *
 *     +--------------------+----------------------------------------+
 *     |  Component         |  Responsibility                        |
 *     +--------------------+----------------------------------------+
 *     |  Storage Engine    |  How data is stored and read via       |
 *     |                    |  pages and blocks on disk              |
 *     +--------------------+----------------------------------------+
 *     |  Query Processor   |  Parses SQL, builds execution plan     |
 *     +--------------------+----------------------------------------+
 *     |  Query Optimizer   |  Makes the plan as fast as possible    |
 *     |                    |  using indexes and statistics          |
 *     +--------------------+----------------------------------------+
 *     |  Transaction Mgr   |  Enforces ACID properties              |
 *     +--------------------+----------------------------------------+
 *     |  Lock Manager      |  Controls concurrent access to data    |
 *     +--------------------+----------------------------------------+
 *     |  Buffer Pool       |  Caches hot pages in RAM               |
 *     |                    |  avoids repeated disk reads            |
 *     +--------------------+----------------------------------------+
 *     |  WAL               |  Logs changes before applying them     |
 *     |                    |  enables crash recovery                |
 *     +--------------------+----------------------------------------+
 *     |  Metadata Store    |  Stores schema, indexes, statistics,   |
 *     |                    |  permissions — database's own DB       |
 *     +--------------------+----------------------------------------+
 */


In [ ]:
/**
 * ============================================================
 * ACID Properties — The Four Guarantees of a Database
 * ============================================================
 *
 *   ACID is not a feature you turn on — it is a set of four
 *   properties that every reliable database must guarantee
 *   for every transaction.
 *
 *   Without ACID, your data can become corrupted, inconsistent,
 *   or permanently lost — even if your application code is perfect.
 *
 *
 *   What is a Transaction?
 *   - Before understanding ACID, understand what a transaction is.
 *   - A transaction is a group of one or more database operations
 *     that are treated as a single unit of work.
 *   - Either ALL operations in the group succeed together,
 *     or NONE of them happen.
 *   - A transaction always ends with either COMMIT (save) or
 *     ROLLBACK (undo everything).
 *
 *    Transaction lifecycle:
 *
 *    BEGIN TRANSACTION
 *         │
 *         ├── Operation 1: Deduct ₹1000 from Hitesh
 *         ├── Operation 2: Add ₹1000 to Piyush
 *         ├── Operation 3: Insert row in transfer_log
 *         │
 *         ▼
 *    All succeed? → COMMIT   → changes saved permanently ✓
 *    Any failure? → ROLLBACK → all changes undone ✓
 *
 *
 *   ACID stands for:
 *
 *     A — Atomicity
 *     C — Consistency
 *     I — Isolation
 *     D — Durability
 *
 *
 * ───────────────────────────────────────────────────────
 * A — Atomicity
 * ───────────────────────────────────────────────────────
 *
 *   Definition:
 *   A transaction is atomic — meaning it is indivisible.
 *   Either ALL operations inside it succeed, or NONE of
 *   them happen. There is no partial success.
 *
 *   The word "atomic" comes from the Greek word for
 *   "indivisible" — something that cannot be cut in half.
 *
 *
 *   Real-world example — bank transfer:
 *
 *   Transfer ₹1000 from Hitesh to Piyush.
 *   Two operations must happen together:
 *   1. Deduct ₹1000 from Hitesh's account
 *   2. Add ₹1000 to Piyush's account
 *
 *
 *   WITHOUT Atomicity (disaster):
 *
 *    BEGIN
 *    Step 1: Deduct ₹1000 from Hitesh  → SUCCESS ✓
 *    Step 2: Add ₹1000 to Piyush       → SERVER CRASHES ✗
 *
 *    Result:
 *    Hitesh  : ₹10,000 → ₹9,000   (₹1000 gone)
 *    Piyush  : ₹5,000  → ₹5,000   (nothing received)
 *    ₹1000 has vanished from the system ✗
 *
 *
 *   WITH Atomicity (safe):
 *
 *    BEGIN TRANSACTION
 *    Step 1: Deduct ₹1000 from Hitesh  → SUCCESS ✓
 *    Step 2: Add ₹1000 to Piyush       → SERVER CRASHES ✗
 *    ← crash detected on restart
 *    ROLLBACK — Step 1 is reversed
 *
 *    Result:
 *    Hitesh  : ₹10,000 (restored) ✓
 *    Piyush  : ₹5,000  (unchanged) ✓
 *    Total money in system: unchanged ✓
 *
 *
 *   VISUAL — Atomicity: all or nothing:
 *
 *    ┌──────────────────────────────────────────────────┐
 *    │  TRANSACTION                                     │
 *    │                                                  │
 *    │  ┌────────┐  ┌────────┐  ┌────────┐              │
 *    │  │ Step 1 │  │ Step 2 │  │ Step 3 │              │
 *    │  │  ✓     │  │  ✓     │  │  ✓    │   → COMMIT   │
 *    │  └────────┘  └────────┘  └────────┘              │
 *    │                                                  │
 *    │  ┌────────┐  ┌────────┐  ┌────────┐              │
 *    │  │ Step 1 │  │ Step 2 │  │ Step 3 │              │
 *    │  │  ✓     │  │  ✗    │  │  ?     │   → ROLLBACK │
 *    │  └────────┘  └────────┘  └────────┘              │
 *    │  Step 1 is also undone — even though it succeeded│
 *    └──────────────────────────────────────────────────┘
 *
 *   How the database achieves Atomicity:
 *   - Using the WAL (Write-Ahead Log).
 *   - Every operation in a transaction is written to the log.
 *   - On crash, the database reads the log:
 *     - If COMMIT is in the log → replay all operations ✓
 *     - If COMMIT is NOT in the log → rollback all operations ✓
 *
 *
 * ───────────────────────────────────────────────────────
 * C — Consistency
 * ───────────────────────────────────────────────────────
 *
 *   Definition:
 *   A transaction must bring the database from one valid
 *   state to another valid state. It can never leave the
 *   database in an invalid or broken state.
 *
 *   "Valid state" means all rules, constraints, and
 *   relationships defined in the schema are satisfied —
 *   before the transaction and after the transaction.
 *
 *
 *   What counts as a "rule" or "constraint"?
 *
 *     +---------------------+------------------------------------+
 *     |  Constraint type    |  Example                           |
 *     +---------------------+------------------------------------+
 *     |  NOT NULL           |  email cannot be empty             |
 *     |  UNIQUE             |  no two users share same email     |
 *     |  CHECK              |  age must be between 0 and 120     |
 *     |  FOREIGN KEY        |  orders.user_id must exist in      |
 *     |                     |  users.id                          |
 *     |  Business rule      |  account balance cannot go         |
 *     |  (app level)        |  below zero                        |
 *     +---------------------+------------------------------------+
 *
 *
 *   Example — Consistency in action:
 *
 *    Rule: account balance cannot go below zero.
 *
 *    Hitesh has ₹500. Someone tries to transfer ₹1000.
 *
 *    BEGIN TRANSACTION
 *    Step 1: Deduct ₹1000 from Hitesh (balance = -₹500)
 *    ← Consistency check: balance < 0 → RULE VIOLATED
 *    ROLLBACK — entire transaction cancelled ✓
 *
 *    Result:
 *    Hitesh stays at ₹500 ✓
 *    Database remains in a valid state ✓
 *
 *
 *   VISUAL — Valid state → Valid state:
 *
 *    Before transaction:          After transaction:
 *    ┌──────────────────┐         ┌──────────────────┐
 *    │ Valid State A    │         │ Valid State B    │
 *    │ Hitesh: ₹10,000  │  ─────► │ Hitesh: ₹9,000   │
 *    │ Piyush: ₹5,000   │         │ Piyush: ₹6,000   │
 *    │ Total:  ₹15,000  │         │ Total:  ₹15,000  │
 *    │ (all rules met)  │         │ (all rules met)  │
 *    └──────────────────┘         └──────────────────┘
 *    Total money is preserved — no money created or lost ✓
 *
 *
 *   Important distinction — Consistency vs Atomicity:
 *   - Atomicity ensures operations are all-or-nothing.
 *   - Consistency ensures the result satisfies all rules.
 *   - They work together: Atomicity prevents partial updates,
 *     Consistency ensures the full update is valid.
 *   - Consistency is partly enforced by the database (schema
 *     constraints) and partly by your application logic
 *     (business rules like "balance cannot go below zero").
 *
 *
 * ───────────────────────────────────────────────────────
 * I — Isolation
 * ───────────────────────────────────────────────────────
 *
 *   Definition:
 *   Concurrent transactions must not interfere with each other.
 *   Each transaction should behave as if it is the only
 *   transaction running — even when thousands are running
 *   simultaneously.
 *
 *   Isolation is the most complex of the four ACID properties
 *   because it involves multiple transactions interacting.
 *
 *
 *   What goes wrong without Isolation:
 *
 *   Problem 1 — Dirty Read:
 *   Transaction A reads data that Transaction B has modified
 *   but NOT yet committed. If B rolls back, A has read data
 *   that never officially existed.
 *
 *    Transaction A              Transaction B
 *    ─────────────              ─────────────
 *                               BEGIN
 *                               UPDATE balance = 9000
 *                               (not committed yet)
 *    BEGIN
 *    SELECT balance → 9000  ← dirty read ✗
 *    (reads B's uncommitted change)
 *                               ROLLBACK (balance = 10000 again)
 *    A acts on a value that
 *    never actually existed ✗
 *
 *
 *   Problem 2 — Non-Repeatable Read:
 *   Transaction A reads the same row twice.
 *   Between the two reads, Transaction B modifies that row.
 *   A gets different values for the same row in one transaction.
 *
 *    Transaction A              Transaction B
 *    ─────────────              ─────────────
 *    BEGIN
 *    SELECT balance → 10000
 *                               BEGIN
 *                               UPDATE balance = 9000
 *                               COMMIT ✓
 *    SELECT balance → 9000  ← different value! ✗
 *    Same query, same transaction, different result ✗
 *
 *
 *   Problem 3 — Phantom Read:
 *   Transaction A runs a query that returns a set of rows.
 *   Between two identical queries, Transaction B inserts
 *   new rows that match A's query. A sees "phantom" rows.
 *
 *    Transaction A              Transaction B
 *    ─────────────              ─────────────
 *    BEGIN
 *    SELECT COUNT(*) WHERE age > 25 → 100 rows
 *                               BEGIN
 *                               INSERT 5 users with age > 25
 *                               COMMIT ✓
 *    SELECT COUNT(*) WHERE age > 25 → 105 rows ← phantom ✗
 *    Where did 5 extra rows come from? ✗
 *
 *
 *   Isolation Levels — the trade-off between safety and speed:
 *
 *   - Full isolation (serialisable) = safest but slowest.
 *   - Lower isolation = faster but allows some anomalies.
 *   - Most databases let you choose your isolation level
 *     based on how much consistency you need vs how much
 *     performance you can sacrifice.
 *
 *     +--------------------+----------+----------+----------+--------+
 *     |  Isolation Level   |  Dirty   |  Non-Rep | Phantom  | Speed  |
 *     |                    |  Read    |  Read    |  Read    |        |
 *     +--------------------+----------+----------+----------+--------+
 *     |  Read Uncommitted  |  ✗ YES   |  ✗ YES   |  ✗ YES   | Fastest|
 *     |  Read Committed    |  ✓ NO    |  ✗ YES   |  ✗ YES   | Fast   |
 *     |  Repeatable Read   |  ✓ NO    |  ✓ NO    |  ✗ YES   | Medium |
 *     |  Serialisable      |  ✓ NO    |  ✓ NO    |  ✓ NO    | Slowest|
 *     +--------------------+----------+----------+----------+--------+
 *     ✓ = problem prevented   ✗ = problem can occur
 *
 *   Default isolation level:
 *   - MySQL (InnoDB): Repeatable Read
 *   - PostgreSQL:     Read Committed
 *   - SQL Server:     Read Committed
 *
 *
 *   VISUAL — Isolation: transactions in their own bubbles:
 *
 *    Without Isolation:              With Isolation:
 *
 *    Txn A ───────────────────       Txn A ┌──────────────┐
 *              ↕ interfere                 │ own snapshot │
 *    Txn B ───────────────────       Txn B └──────────────┘
 *              ↕ interfere                 ┌──────────────┐
 *    Txn C ───────────────────             │ own snapshot │
 *    Reads each other's                    └──────────────┘
 *    uncommitted data ✗           Each transaction sees a
 *                                 consistent view ✓
 *
 *   How databases achieve Isolation:
 *   - Locking: transactions acquire locks on rows/pages
 *     they are reading or writing. Others must wait.
 *   - MVCC (Multi-Version Concurrency Control): the database
 *     keeps multiple versions of a row simultaneously.
 *     Each transaction reads the version that existed when
 *     it started — without blocking other transactions.
 *     PostgreSQL and MySQL InnoDB both use MVCC.
 *
 *
 * ───────────────────────────────────────────────────────
 * D — Durability
 * ───────────────────────────────────────────────────────
 *
 *   Definition:
 *   Once a transaction is committed, it is permanent.
 *   Even if the server crashes, the power goes out, or
 *   the disk has an error — a committed transaction must
 *   survive. It will never be lost.
 *
 *   This is the "D" in ACID and it is directly enabled
 *   by the WAL (Write-Ahead Log) we discussed in the
 *   Core Components section.
 *
 *
 *   What "committed" means:
 *   - When the database says COMMIT, it means the WAL log
 *     entry for this transaction has been flushed to disk.
 *   - At that point, the data is durable — even if it is
 *     still only in the Buffer Pool (RAM) and not yet
 *     written to the actual data file.
 *   - On restart, the WAL ensures it gets replayed.
 *
 *
 *   VISUAL — Durability guarantee:
 *
 *    Your app:  UPDATE users SET age = 26 WHERE id = 5
 *    Database:  COMMIT
 *         │
 *         │  Server crashes 1 millisecond after COMMIT
 *         ▼
 *    Server restarts
 *         │
 *         ▼
 *    Database reads WAL:
 *    "LSN 1001: UPDATE users id=5 age→26 — COMMITTED"
 *         │
 *         ▼
 *    Replays the change ✓
 *    age = 26 is permanently saved ✓
 *    Durability maintained ✓
 *
 *
 *   Without Durability (disaster):
 *
 *    Database says COMMIT
 *    Data is in RAM (Buffer Pool) but not yet on disk
 *    Server crashes
 *    RAM is wiped
 *    No WAL → no way to recover
 *    Data permanently lost ✗
 *    Your app thought it was saved — it was not ✗
 *
 *
 *   How Durability is implemented:
 *
 *    ┌──────────────────────────────────────────────────┐
 *    │  COMMIT received                                 │
 *    │       │                                          │
 *    │       ▼                                          │
 *    │  WAL log entry flushed to disk (fsync)           │
 *    │  ← this is the durability guarantee point        │
 *    │       │                                          │
 *    │       ▼                                          │
 *    │  Return "COMMIT successful" to application       │
 *    │       │                                          │
 *    │       ▼  (happens later, asynchronously)         │
 *    │  Data page written from Buffer Pool to disk      │
 *    │  (checkpoint)                                    │
 *    └──────────────────────────────────────────────────┘
 *
 *   Note — fsync:
 *   - fsync is a system call that forces the OS to flush
 *     its write buffers to the physical disk.
 *   - Without fsync, data might sit in the OS cache and
 *     appear written but crash before hitting the disk.
 *   - Databases call fsync on the WAL log to guarantee
 *     durability at the OS level.
 *   - This is also why turning off fsync (a dangerous
 *     performance trick) breaks the Durability guarantee.
 *
 *
 * ───────────────────────────────────────────────────────
 * ACID Together — How All Four Work as One System
 * ───────────────────────────────────────────────────────
 *
 *   ACID is not four separate features. They are four
 *   properties of one unified system and they depend on
 *   each other.
 *
 *
 *   VISUAL — ACID dependencies:
 *
 *    ┌─────────────────────────────────────────────────────────┐
 *    │                                                         │
 *    │   Atomicity  ──────────────────────────────────────┐    │
 *    │   (all or nothing)                                 │    │
 *    │                                                    │    │
 *    │   Consistency ─────────────────────────────────┐   │    │
 *    │   (valid state to valid state)                 │   │    │
 *    │                                                │   │    │
 *    │   Isolation ───────────────────────────────┐   │   │    │
 *    │   (no interference between transactions)   │   │   │    │
 *    │                                            │   │   │    │
 *    │   Durability ───────────────────────────┐  │   │   │    │
 *    │   (committed = permanent)               │  │   │   │    │
 *    │                                         ▼  ▼   ▼   ▼    │
 *    │                                    ┌─────────────────┐  │
 *    │                                    │  WAL + Locking  │  │
 *    │                                    │  + MVCC         │  │
 *    │                                    │  (implementation│  │
 *    │                                    │   mechanisms)   │  │
 *    │                                    └─────────────────┘  │
 *    └─────────────────────────────────────────────────────────┘
 *
 *
 *   Complete bank transfer example showing all four:
 *
 *    Transfer ₹1000 from Hitesh (₹10,000) to Piyush (₹5,000)
 *
 *    BEGIN TRANSACTION
 *    ┌─────────────────────────────────────────────────────────┐
 *    │                                                         │
 *    │  [ISOLATION]                                            │
 *    │  Other transactions cannot see these changes yet        │
 *    │  Hitesh's balance appears as ₹10,000 to everyone else   │
 *    │                                                         │
 *    │  Step 1: Deduct ₹1000 from Hitesh → balance = ₹9,000    │
 *    │  Step 2: Add ₹1000 to Piyush    → balance = ₹6,000      │
 *    │  Step 3: Log entry in transfers table                   │
 *    │                                                         │
 *    │  [CONSISTENCY check]                                    │
 *    │  Hitesh balance ≥ 0? YES ✓                              │
 *    │  Piyush is a valid user? YES ✓                          │
 *    │  Total money unchanged? YES ✓                           │
 *    │                                                         │
 *    │  [ATOMICITY check]                                      │
 *    │  All 3 steps succeeded? YES ✓                           │
 *    │                                                         │
 *    └─────────────────────────────────────────────────────────┘
 *    COMMIT
 *    │
 *    │  [DURABILITY]
 *    │  WAL log flushed to disk
 *    │  "COMMITTED" returned to application
 *    │  Changes now visible to all other transactions ✓
 *    ▼
 *    Transfer complete ✓
 *    Hitesh: ₹9,000 | Piyush: ₹6,000 | Total: ₹15,000 ✓
 *
 *
 *   Summary — ACID quick reference:
 *
 *     +------------------+-------------------------------------------+
 *     |  Property        |  One-line summary                         |
 *     +------------------+-------------------------------------------+
 *     |  Atomicity       |  All steps succeed or nothing happens     |
 *     +------------------+-------------------------------------------+
 *     |  Consistency     |  Data always satisfies all defined rules  |
 *     +------------------+-------------------------------------------+
 *     |  Isolation       |  Transactions don't see each other's      |
 *     |                  |  uncommitted work                         |
 *     +------------------+-------------------------------------------+
 *     |  Durability      |  Committed data survives any crash        |
 *     +------------------+-------------------------------------------+
 *     |  Implemented by  |  WAL (A + D) + Locking/MVCC (I) +         |
 *     |                  |  Schema constraints (C)                   |
 *     +------------------+-------------------------------------------+
 */


In [ ]:
/**
 * ============================================================
 * CAP Theorem
 * ============================================================
 *
 *   CAP Theorem states that in a distributed database system,
 *   you can only guarantee TWO out of these three properties
 *   at the same time — never all three.
 *
 *     C — Consistency
 *     A — Availability
 *     P — Partition Tolerance
 *
 *
 * ───────────────────────────────────────────────────────
 * CONCEPT 1 — What is a Distributed Database?
 * ───────────────────────────────────────────────────────
 *
 *   - A single database running on one machine is not distributed.
 *   - A distributed database runs on multiple machines (nodes)
 *     connected over a network.
 *   - Each node has its own IP address.
 *   - Data is replicated or split across these nodes.
 *   - This is done for scale, fault tolerance, and availability.
 *
 *
 *   VISUAL — Single node vs distributed:
 *
 *    Single node:                  Distributed:
 *    ┌──────────────┐              ┌──────────┐  ┌──────────┐
 *    │   Database   │              │  Node 1  │  │  Node 2  │
 *    │  192.168.1.1 │              │ 10.0.0.1 │  │ 10.0.0.2 │
 *    └──────────────┘              └────┬─────┘  └────┬─────┘
 *    One IP, one machine                └──────┬───────┘
 *                                         network
 *                                       ┌──────┴──────┐
 *                                       │   Node 3    │
 *                                       │  10.0.0.3   │
 *                                       └─────────────┘
 *    Once you have multiple nodes → CAP Theorem applies.
 *
 *
 * ───────────────────────────────────────────────────────
 * CONCEPT 2 — The Three Properties Explained
 * ───────────────────────────────────────────────────────
 *
 *   Property 1 — Consistency (C):
 *   - Every read from ANY node returns the most recent write
 *     or an error. No stale data, ever.
 *   - All nodes see the same data at the same time.
 *   - This is NOT the same C as in ACID. ACID Consistency is
 *     about rules and constraints. CAP Consistency is about
 *     all nodes having the same data at the same moment.
 *
 *
 *   VISUAL — CAP Consistency:
 *
 *    User writes: name = "Hitesh" → Node 1
 *         │
 *         ▼
 *    Node 1 syncs to Node 2 and Node 3 immediately
 *
 *    User reads from Node 2 → "Hitesh" ✓
 *    User reads from Node 3 → "Hitesh" ✓
 *
 *    All nodes return the same value always ✓
 *
 *
 *   Property 2 — Availability (A):
 *   - Every request to a non-failing node gets a response.
 *   - The response may not be the most recent data —
 *     but you always get A response (not an error).
 *   - The system never refuses to answer.
 *
 *
 *   VISUAL — Availability:
 *
 *    Node 2 is out of sync with Node 1
 *    User reads from Node 2
 *         │
 *         ▼
 *    Node 2 returns its own (possibly stale) data
 *    "Hitesh" from 5 minutes ago → returned ✓
 *    (might be wrong but never returns an error)
 *
 *    System is always responsive ✓
 *    Data might be stale ✗
 *
 *
 *   Property 3 — Partition Tolerance (P):
 *   - A partition is when the network link between nodes
 *     breaks — nodes cannot talk to each other.
 *   - Partition Tolerance means the system continues to
 *     work even when a network partition occurs.
 *   - The system does not crash just because two nodes
 *     cannot communicate temporarily.
 *
 *
 *   VISUAL — Network Partition:
 *
 *    Normal:                       Partition (network split):
 *    ┌───────┐   ←──────►  ┌───────┐    ┌───────┐    ✗    ┌───────┐
 *    │ Node1 │   network    │ Node2 │    │ Node1 │─ ─ ─ ─ ─│ Node2 │
 *    └───────┘   works      └───────┘    └───────┘  broken  └───────┘
 *    Nodes sync continuously            Nodes cannot talk to each other
 *                                       What does the system do?
 *
 *
 * ───────────────────────────────────────────────────────
 * CONCEPT 3 — Why You Cannot Have All Three
 * ───────────────────────────────────────────────────────
 *
 *   Imagine a distributed database with two nodes:
 *   Node 1 (Mumbai) and Node 2 (Delhi).
 *
 *   A network partition happens — they cannot talk to each other.
 *
 *   A user writes "name = Hitesh" to Node 1.
 *   Another user reads from Node 2.
 *
 *   Now Node 2 has a choice:
 *
 *
 *   Choice A — Be Consistent (sacrifice Availability):
 *   - Node 2 knows it is out of sync with Node 1.
 *   - It refuses to answer: "I cannot confirm I have the
 *     latest data. Please try later."
 *   - Returns an error or timeout to the user.
 *   - Result: Consistent ✓ but NOT Available ✗
 *
 *
 *   Choice B — Be Available (sacrifice Consistency):
 *   - Node 2 returns whatever data it has right now.
 *   - It might return "name = Piyush" (old data) even though
 *     Node 1 now has "name = Hitesh".
 *   - Result: Available ✓ but NOT Consistent ✗
 *
 *
 *   VISUAL — The impossible triangle:
 *
 *                      Consistency (C)
 *                           ▲
 *                          /│\
 *                         / │ \
 *                        /  │  \
 *                       /   │   \
 *                      /    │    \
 *                     /  ✗  │  ✗  \
 *                    /  CAC │ CPC  \
 *                   /───────────────\
 *    Availability (A)─────────────── Partition Tolerance (P)
 *
 *    You can only pick 2 of the 3 vertices.
 *    The third is always sacrificed.
 *
 *
 * ───────────────────────────────────────────────────────
 * CONCEPT 4 — Partition Tolerance is NOT Optional
 * ───────────────────────────────────────────────────────
 *
 *   - In theory, CAP gives you three combinations to choose from:
 *     CP, AP, or CA.
 *   - In reality, if you are running a distributed system,
 *     you MUST have Partition Tolerance. It is not optional.
 *   - Why? Because network failures WILL happen in the real world.
 *     Cables get cut, switches fail, data centers lose connectivity.
 *     You cannot simply choose to not handle network failures.
 *   - So the real choice in a distributed system is only:
 *     CP or AP — do you sacrifice Consistency or Availability
 *     when a partition happens?
 *
 *
 *   VISUAL — The real-world CAP choice:
 *
 *    In a distributed system:
 *
 *    Partition Tolerance = MANDATORY (non-negotiable)
 *
 *    Real choice:
 *    ┌──────────────────────────────────────────────────┐
 *    │                                                  │
 *    │   CP — Consistent + Partition Tolerant           │
 *    │   When partition happens → refuse to answer      │
 *    │   (sacrifice Availability)                       │
 *    │                                                  │
 *    │              OR                                  │
 *    │                                                  │
 *    │   AP — Available + Partition Tolerant            │
 *    │   When partition happens → return stale data     │
 *    │   (sacrifice Consistency)                        │
 *    │                                                  │
 *    └──────────────────────────────────────────────────┘
 *
 *   CA (Consistency + Availability, no Partition Tolerance)
 *   only makes sense for a single-node database that never
 *   splits across a network — like a traditional Postgres
 *   on one machine.
 *
 *
 * ───────────────────────────────────────────────────────
 * CONCEPT 5 — CP vs AP vs CA with Real Examples
 * ───────────────────────────────────────────────────────
 *
 *   CP — Consistency + Partition Tolerance:
 *   (MongoDB, HBase, Redis, Zookeeper)
 *
 *   - When a network partition occurs, the system chooses
 *     to stop responding rather than return stale data.
 *   - Reads and writes to out-of-sync nodes are rejected.
 *   - Once the partition heals, the system syncs and
 *     becomes available again.
 *   - Best for: banking, financial systems, inventory
 *     management — anywhere where wrong data is worse
 *     than no data.
 *
 *
 *   VISUAL — CP behaviour during partition:
 *
 *    ┌──────────┐  ✗ network  ┌──────────┐
 *    │ MongoDB  │─ ─ ─ ─ ─ ─ ─│ MongoDB  │
 *    │ Node 1   │   split     │ Node 2   │
 *    └──────────┘             └──────────┘
 *
 *    User reads from Node 2:
 *    → Node 2: "I cannot confirm I have latest data"
 *    → Returns ERROR / timeout ✓ (never stale data)
 *
 *    Partition heals → nodes sync → reads work again ✓
 *
 *
 *   AP — Availability + Partition Tolerance:
 *   (DynamoDB, Cassandra, CouchDB)
 *
 *   - When a network partition occurs, the system continues
 *     to respond with whatever data it has.
 *   - Nodes may return stale or conflicting data temporarily.
 *   - Once the partition heals, the system reconciles
 *     differences (eventual consistency).
 *   - Best for: social media feeds, shopping carts, DNS,
 *     leaderboards — anywhere where being always available
 *     matters more than perfect accuracy.
 *
 *
 *   VISUAL — AP behaviour during partition:
 *
 *    ┌──────────┐  ✗ network  ┌──────────┐
 *    │ DynamoDB │─ ─ ─ ─ ─ ─ ─│ DynamoDB │
 *    │ Node 1   │   split     │ Node 2   │
 *    └──────────┘             └──────────┘
 *
 *    Node 1 has: name = "Hitesh" (latest)
 *    Node 2 has: name = "Piyush" (stale)
 *
 *    User reads from Node 2:
 *    → Node 2: returns "Piyush" ✓ (wrong but always responds)
 *
 *    Partition heals → nodes reconcile → "Hitesh" wins ✓
 *    (Eventual Consistency)
 *
 *
 *   CA — Consistency + Availability:
 *   (PostgreSQL, MySQL — single node)
 *
 *   - No partition tolerance — only works on a single machine
 *     or a tightly coupled cluster where network failure is
 *     treated as a total system failure.
 *   - Perfect consistency and availability as long as the
 *     single node is running.
 *   - The moment you add a second node over a network,
 *     you are no longer CA — you must pick CP or AP.
 *   - Best for: traditional applications on a single server
 *     where distributed scaling is not needed.
 *
 *
 *   VISUAL — CA on a single node:
 *
 *    ┌──────────────────────────────────────────────┐
 *    │   PostgreSQL (single node)                    │
 *    │                                              │
 *    │   Always consistent ✓                        │
 *    │   Always available ✓                         │
 *    │   No network partition possible ✓            │
 *    │   (only one machine — nothing to split)      │
 *    └──────────────────────────────────────────────┘
 *
 *
 *   Database classification summary:
 *
 *     +------------------+----------+----------------------------------+
 *     |  Database        |  Type    |  Trade-off                       |
 *     +------------------+----------+----------------------------------+
 *     |  MongoDB         |  CP      |  Rejects reads during partition  |
 *     |                  |          |  Never returns stale data        |
 *     +------------------+----------+----------------------------------+
 *     |  HBase           |  CP      |  Strong consistency, may timeout |
 *     +------------------+----------+----------------------------------+
 *     |  Redis           |  CP      |  Master fails → elections happen |
 *     +------------------+----------+----------------------------------+
 *     |  DynamoDB        |  AP      |  Always available, eventual sync |
 *     +------------------+----------+----------------------------------+
 *     |  Cassandra       |  AP      |  Always available, tunable       |
 *     +------------------+----------+----------------------------------+
 *     |  CouchDB         |  AP      |  Available, conflict resolution  |
 *     +------------------+----------+----------------------------------+
 *     |  PostgreSQL      |  CA      |  Single node, no partition       |
 *     +------------------+----------+----------------------------------+
 *     |  MySQL           |  CA      |  Single node, no partition       |
 *     +------------------+----------+----------------------------------+
 *
 *
 * ───────────────────────────────────────────────────────
 * CONCEPT 6 — Eventual Consistency (AP Systems)
 * ───────────────────────────────────────────────────────
 *
 *   - AP systems use a model called Eventual Consistency.
 *   - It means: given enough time with no new writes,
 *     all nodes will eventually converge to the same value.
 *   - It does NOT mean data will always be wrong.
 *   - It means data might be temporarily stale during a
 *     partition but will be correct once the partition heals.
 *
 *
 *   VISUAL — Eventual Consistency timeline:
 *
 *    t=0s:  User writes "name = Hitesh" to Node 1
 *    t=0s:  Node 2 still has "name = Piyush" (stale)
 *           ← partition exists
 *
 *    t=0s:  Read from Node 1 → "Hitesh"  ✓ (latest)
 *    t=0s:  Read from Node 2 → "Piyush"  ✗ (stale)
 *
 *           Network partition heals at t=5s
 *
 *    t=5s:  Node 2 syncs with Node 1
 *    t=5s:  Node 2 now has "name = Hitesh"
 *
 *    t=6s:  Read from Node 1 → "Hitesh"  ✓
 *    t=6s:  Read from Node 2 → "Hitesh"  ✓
 *
 *    Eventually consistent ✓
 *
 *
 * ───────────────────────────────────────────────────────
 * CONCEPT 7 — Choosing CP vs AP
 * ───────────────────────────────────────────────────────
 *
 *   Ask yourself: what is worse for your application —
 *   wrong data or no data?
 *
 *
 *   Choose CP when:
 *   Wrong data is worse than no data.
 *
 *     +-----------------------------+----------------------------------+
 *     |  Use Case                   |  Why CP                          |
 *     +-----------------------------+----------------------------------+
 *     |  Bank account balance       |  Showing wrong balance is        |
 *     |                             |  dangerous. Better to error.     |
 *     +-----------------------------+----------------------------------+
 *     |  Payment processing         |  Double charge is catastrophic.  |
 *     +-----------------------------+----------------------------------+
 *     |  Inventory (last 1 item)    |  Overselling is a business       |
 *     |                             |  disaster.                       |
 *     +-----------------------------+----------------------------------+
 *     |  Authentication tokens      |  Wrong auth = security breach.   |
 *     +-----------------------------+----------------------------------+
 *
 *
 *   Choose AP when:
 *   No data is worse than slightly stale data.
 *
 *     +-----------------------------+----------------------------------+
 *     |  Use Case                   |  Why AP                          |
 *     +-----------------------------+----------------------------------+
 *     |  Social media feed          |  Slightly old posts are fine.    |
 *     |                             |  Not loading is worse.           |
 *     +-----------------------------+----------------------------------+
 *     |  Product recommendations    |  Stale recs are acceptable.      |
 *     +-----------------------------+----------------------------------+
 *     |  Like / view counts         |  Off by a few is fine.           |
 *     +-----------------------------+----------------------------------+
 *     |  Shopping cart              |  Always show cart, sync later.   |
 *     +-----------------------------+----------------------------------+
 *     |  DNS resolution             |  Stale DNS for a few minutes     |
 *     |                             |  is acceptable.                  |
 *     +-----------------------------+----------------------------------+
 *
 *
 *   Decision guide:
 *
 *    Is your system distributed (multiple nodes)?
 *         │
 *    ┌────┴────┐
 *    │         │
 *   YES        NO
 *    │         │
 *    ▼         ▼
 *    P is     CA database
 *    mandatory (Postgres/MySQL
 *    │         single node)
 *    │
 *    ▼
 *    Is wrong data worse than no data?
 *         │
 *    ┌────┴────┐
 *    │         │
 *   YES        NO
 *    │         │
 *    ▼         ▼
 *   CP         AP
 *  (MongoDB)  (DynamoDB)
 *
 *
 *   Summary:
 *
 *     +-------------------+------------------------------------------+
 *     |  Property         |  One-line summary                        |
 *     +-------------------+------------------------------------------+
 *     |  Consistency (C)  |  All nodes return the same data always   |
 *     +-------------------+------------------------------------------+
 *     |  Availability (A) |  System always responds, may be stale   |
 *     +-------------------+------------------------------------------+
 *     |  Partition        |  System works even when nodes can't      |
 *     |  Tolerance (P)    |  communicate (mandatory in distributed)  |
 *     +-------------------+------------------------------------------+
 *     |  CP               |  Consistent but may timeout/error        |
 *     |  (MongoDB)        |  during network split                    |
 *     +-------------------+------------------------------------------+
 *     |  AP               |  Always responds but may return stale    |
 *     |  (DynamoDB)       |  data during network split               |
 *     +-------------------+------------------------------------------+
 *     |  CA               |  Single node only — no distributed split |
 *     |  (PostgreSQL)     |  possible                                |
 *     +-------------------+------------------------------------------+
 */

In [ ]:
/**
 * ============================================================
 * Relational vs Non-Relational Databases
 * ============================================================
 *
 *   This is one of the most debated topics in system design.
 *   There is no universally correct answer — both have their
 *   place, and understanding the trade-offs is what separates
 *   a good engineer from a great one.
 *
 *
 * ───────────────────────────────────────────────────────
 * CONCEPT 1 — How Each Stores Data
 * ───────────────────────────────────────────────────────
 *
 *   Relational databases store data in tables — rows and columns.
 *   Every row in a table has the exact same shape.
 *   Relationships between data are enforced through foreign keys.
 *
 *   Non-relational databases store data in formats that match
 *   the shape of the data itself — documents, key-value pairs,
 *   graphs, or columns. No fixed shape required.
 *
 *
 *   VISUAL — Same user data stored differently:
 *
 *   Relational (PostgreSQL):
 *
 *   users table:
 *   ┌────┬──────────┬─────┬────────┐
 *   │ id │ name     │ age │ city   │
 *   ├────┼──────────┼─────┼────────┤
 *   │  1 │ Hitesh   │  28 │ Mumbai │
 *   │  2 │ Piyush   │  25 │ Delhi  │
 *   └────┴──────────┴─────┴────────┘
 *
 *   user_skills table:
 *   ┌─────────┬──────────┐
 *   │ user_id │ skill    │
 *   ├─────────┼──────────┤
 *   │    1    │ Node.js  │
 *   │    1    │ AWS      │
 *   │    2    │ React    │
 *   └─────────┴──────────┘
 *   Data spread across two tables, joined on demand.
 *
 *   Non-Relational (MongoDB):
 *
 *   users collection:
 *   {                              {
 *     _id: 1,                        _id: 2,
 *     name: "Hitesh",                name: "Piyush",
 *     age: 28,                       age: 25,
 *     city: "Mumbai",                city: "Delhi",
 *     skills: ["Node.js","AWS"]      skills: ["React"]
 *   }                              }
 *   All related data in one document — no JOIN needed.
 *
 *
 * ───────────────────────────────────────────────────────
 * CONCEPT 2 — Schema: Rigid vs Flexible
 * ───────────────────────────────────────────────────────
 *
 *   Schema = the structure and rules your data must follow.
 *
 *   Relational — Schema First (rigid):
 *   - You define the schema before writing any data.
 *   - Every row MUST follow that schema.
 *   - Adding a new field requires a migration — altering the
 *     table, which can lock the table and cause downtime
 *     on a large production database.
 *   - Great for structured, well-understood data.
 *   - Bad for rapidly changing data requirements.
 *
 *
 *   VISUAL — Adding a field in SQL vs NoSQL:
 *
 *   Relational (painful at scale):
 *   ALTER TABLE users ADD COLUMN linkedin VARCHAR(200);
 *   ← This locks the table for the duration of the migration
 *   ← On a table with 100 million rows: could take hours ✗
 *   ← App is slow or unavailable during this time ✗
 *
 *   Non-Relational (effortless):
 *   Just start writing the new field in new documents.
 *   db.users.updateOne({ _id:1 }, { $set: { linkedin:"..." } })
 *   Old docs don't have it — that is fine ✓
 *   New docs have it — that is also fine ✓
 *   No migration, no downtime, no locking ✓
 *
 *
 *   Non-Relational — Schema on Read (flexible):
 *   - You do not define the schema upfront.
 *   - Each document can have completely different fields.
 *   - The application handles the shape of data when it reads.
 *   - Great for evolving data models and rapid development.
 *   - Dangerous if not disciplined — garbage data can sneak in.
 *
 *
 *   VISUAL — Flexible schema in practice:
 *
 *   All three documents live in the same collection:
 *
 *   Doc 1 (basic user):
 *   { _id:1, name:"Hitesh", age:28 }
 *
 *   Doc 2 (user with social profiles):
 *   { _id:2, name:"Piyush", age:25, twitter:"@piyush", github:"piyush" }
 *
 *   Doc 3 (user with address):
 *   { _id:3, name:"Aman", age:30, address:{ city:"Pune", zip:"411001" } }
 *
 *   All valid. No errors. No NULL columns. ✓
 *   In SQL: all three users would need ALL columns — mostly NULLs. ✗
 *
 *
 * ───────────────────────────────────────────────────────
 * CONCEPT 3 — Relationships and JOINs
 * ───────────────────────────────────────────────────────
 *
 *   This is where relational databases shine.
 *
 *   Relational — JOINs are first-class citizens:
 *   - Relationships between tables are enforced by the database.
 *   - Foreign keys guarantee referential integrity:
 *     you cannot create an order for a user that does not exist.
 *   - JOINs are efficient, indexed, and optimised by the
 *     Query Optimizer.
 *   - You can query across any combination of tables at runtime
 *     without planning it in advance.
 *
 *
 *   VISUAL — Relational JOIN:
 *
 *   users          orders           products
 *   ┌──────┐       ┌──────────┐     ┌───────────┐
 *   │id    │◄──────│user_id   │     │id         │
 *   │name  │       │product_id│────►│name       │
 *   └──────┘       │amount    │     │price      │
 *                  └──────────┘     └───────────┘
 *
 *   SELECT u.name, p.name, o.amount
 *   FROM orders o
 *   JOIN users    u ON o.user_id    = u.id
 *   JOIN products p ON o.product_id = p.id
 *   WHERE u.name = 'Hitesh'
 *
 *   Three tables joined at query time — no pre-planning ✓
 *   Foreign keys prevent orphan orders ✓
 *
 *
 *   Non-Relational — Two strategies for relationships:
 *
 *   Strategy A — Embed (denormalise):
 *   Put related data inside the document.
 *   Best when data is always read together.
 *
 *   {
 *     _id: "order-001",
 *     user: { id:"u1", name:"Hitesh" },    ← embedded
 *     product: { id:"p1", name:"JS Course" }, ← embedded
 *     amount: 500
 *   }
 *
 *   One read gets everything ✓
 *   User name stored in every order — duplicated ✗
 *   If user changes their name → must update every order ✗
 *
 *
 *   Strategy B — Reference (manual JOIN):
 *   Store only the ID, look up separately.
 *   Best when referenced data changes frequently.
 *
 *   {
 *     _id: "order-001",
 *     userId: "u1",      ← just an ID
 *     productId: "p1",   ← just an ID
 *     amount: 500
 *   }
 *
 *   No duplication ✓
 *   Requires two separate queries (or $lookup) ✗
 *   No foreign key enforcement — orphan records possible ✗
 *
 *
 *   VISUAL — Embed vs Reference trade-off:
 *
 *   ┌──────────────────────────────────────────────────────┐
 *   │              Embed                Reference          │
 *   ├──────────────────────────────────────────────────────┤
 *   │  Reads      Fast (1 query)    Slower (2 queries)     │
 *   │  Updates    Slow (many docs)  Fast (1 doc)           │
 *   │  Duplication Yes              No                     │
 *   │  Consistency Manual           Manual                 │
 *   │  Best for   Read-heavy        Write-heavy /          │
 *   │             static data       frequently changing    │
 *   └──────────────────────────────────────────────────────┘
 *
 *
 * ───────────────────────────────────────────────────────
 * CONCEPT 4 — Scaling
 * ───────────────────────────────────────────────────────
 *
 *   This is the most critical difference at large scale.
 *
 *   Relational — Vertical Scaling (scale up):
 *   - Add more CPU, RAM, and disk to one powerful machine.
 *   - Simple — no data distribution to manage.
 *   - Has a hard ceiling — you cannot add infinite RAM.
 *   - Most powerful single server: ~100k queries/second.
 *   - Read replicas can help with read load.
 *   - Sharding is possible but extremely complex with SQL.
 *
 *
 *   VISUAL — Vertical vs Horizontal scaling:
 *
 *   Vertical (Relational):          Horizontal (Non-Relational):
 *
 *   ┌───────────────┐               ┌───────┐ ┌───────┐ ┌───────┐
 *   │  Big Machine  │               │ Node 1│ │ Node 2│ │ Node 3│
 *   │  32 CPU cores │               │ small │ │ small │ │ small │
 *   │  512 GB RAM   │               └───────┘ └───────┘ └───────┘
 *   │  10 TB SSD    │                    add more nodes as needed
 *   └───────────────┘               ┌───────┐ ┌───────┐
 *   Bigger machine costs $$$        │ Node 4│ │ Node 5│ ...
 *   Hard ceiling ✗                  └───────┘ └───────┘
 *                                   Cheap commodity hardware ✓
 *                                   No ceiling ✓
 *
 *   Non-Relational — Horizontal Scaling (scale out):
 *   - Add more machines — each handles a portion of data.
 *   - Data is automatically distributed (sharded) across nodes.
 *   - Built from the ground up for distributed deployment.
 *   - Can handle millions of reads/writes per second.
 *   - Trade-off: gives up some consistency (CAP theorem).
 *
 *
 *   VISUAL — Sharding (how NoSQL distributes data):
 *
 *   User data split by user_id range across 3 nodes:
 *
 *   ┌──────────────────┐
 *   │  Node 1          │   user_id 1–1,000,000
 *   │  users: 1M docs  │
 *   └──────────────────┘
 *   ┌──────────────────┐
 *   │  Node 2          │   user_id 1,000,001–2,000,000
 *   │  users: 1M docs  │
 *   └──────────────────┘
 *   ┌──────────────────┐
 *   │  Node 3          │   user_id 2,000,001–3,000,000
 *   │  users: 1M docs  │
 *   └──────────────────┘
 *
 *   Query for user_id=500,000 → goes directly to Node 1 ✓
 *   Query for user_id=1,500,000 → goes directly to Node 2 ✓
 *   No single machine bottleneck ✓
 *
 *
 * ───────────────────────────────────────────────────────
 * CONCEPT 5 — ACID vs BASE
 * ───────────────────────────────────────────────────────
 *
 *   Relational databases guarantee ACID.
 *   Most Non-Relational databases follow BASE instead.
 *
 *   BASE stands for:
 *   BA — Basically Available    (system is always up)
 *   S  — Soft state             (data may change without input)
 *   E  — Eventually consistent  (all nodes agree, given time)
 *
 *
 *   VISUAL — ACID vs BASE:
 *
 *   ACID (Relational):               BASE (Non-Relational):
 *   ┌──────────────────────────┐     ┌──────────────────────────┐
 *   │  Strong consistency      │     │  Eventual consistency     │
 *   │  All nodes always agree  │     │  Nodes agree eventually   │
 *   │  Transaction either      │     │  System stays available   │
 *   │  fully commits or rolls  │     │  even during failures     │
 *   │  back — no middle ground │     │  Some stale data is ok    │
 *   │                          │     │                          │
 *   │  Safer for money/data    │     │  Better for scale/speed  │
 *   │  Slower at massive scale │     │  Fine for social/feeds   │
 *   └──────────────────────────┘     └──────────────────────────┘
 *
 *     +------------------+--------------------+--------------------+
 *     |  Property        |  ACID (Relational) |  BASE (NoSQL)      |
 *     +------------------+--------------------+--------------------+
 *     |  Consistency     |  Strong always     |  Eventual          |
 *     |  Availability    |  May sacrifice     |  Always available  |
 *     |  Transactions    |  Full ACID         |  Limited / none    |
 *     |  Best for        |  Financial, ERP    |  Social, analytics |
 *     +------------------+--------------------+--------------------+
 *
 *
 * ───────────────────────────────────────────────────────
 * CONCEPT 6 — Query Language
 * ───────────────────────────────────────────────────────
 *
 *   Relational — SQL (Structured Query Language):
 *   - Standardised — same SQL works on MySQL, PostgreSQL,
 *     Oracle, SQL Server (with minor differences).
 *   - Declarative — you say WHAT you want, not HOW.
 *   - Extremely powerful for complex queries, aggregations,
 *     window functions, CTEs.
 *   - Every developer already knows SQL.
 *
 *   SELECT u.name, COUNT(o.id) as order_count, SUM(o.amount) as total
 *   FROM users u
 *   LEFT JOIN orders o ON u.id = o.user_id
 *   WHERE u.created_at > '2024-01-01'
 *   GROUP BY u.name
 *   HAVING COUNT(o.id) > 5
 *   ORDER BY total DESC
 *   LIMIT 10;
 *
 *
 *   Non-Relational — Database-specific APIs:
 *   - No standard — every database has its own query language.
 *   - MongoDB uses BSON filter objects + aggregation pipelines.
 *   - DynamoDB uses its own API with very limited query options.
 *   - Redis uses its own command set (GET, SET, ZADD, etc.)
 *   - Learning curve is different for each database.
 *
 *   // MongoDB
 *   db.users.aggregate([
 *     { $match: { created_at: { $gt: new Date('2024-01-01') } } },
 *     { $lookup: { from:"orders", localField:"_id",
 *                  foreignField:"userId", as:"orders" } },
 *     { $project: { name:1, order_count:{ $size:"$orders" },
 *                   total:{ $sum:"$orders.amount" } } },
 *     { $match: { order_count: { $gt: 5 } } },
 *     { $sort:  { total: -1 } },
 *     { $limit: 10 }
 *   ])
 *
 *   Same result — but longer, less readable, and only
 *   works on MongoDB. SQL version works everywhere. ✓
 *
 *
 * ───────────────────────────────────────────────────────
 * CONCEPT 7 — Data Integrity and Constraints
 * ───────────────────────────────────────────────────────
 *
 *   Relational — enforced by the database:
 *   - NOT NULL: field cannot be empty
 *   - UNIQUE: no two rows can have the same value
 *   - FOREIGN KEY: referenced record must exist
 *   - CHECK: value must satisfy a condition
 *   - The database REJECTS data that breaks any rule.
 *   - Data quality is guaranteed at the storage layer.
 *
 *
 *   VISUAL — Foreign key enforcement:
 *
 *   INSERT INTO orders (user_id, amount) VALUES (9999, 500);
 *   ← user_id 9999 does not exist in users table
 *   → ERROR: insert or update on table "orders" violates
 *     foreign key constraint — user_id 9999 not in users ✓
 *   Orphan records impossible at the DB level ✓
 *
 *
 *   Non-Relational — enforced by the application:
 *   - The database accepts anything you write.
 *   - Your application code must enforce rules.
 *   - If you forget a validation → bad data silently enters.
 *   - Mongoose (ODM for MongoDB) adds schema validation
 *     at the application level — but it is optional.
 *
 *
 *   VISUAL — No referential integrity in NoSQL:
 *
 *   db.orders.insertOne({ userId:"u9999", amount:500 });
 *   ← user u9999 does not exist in users collection
 *   → Inserted successfully ✗ (orphan record, no error)
 *   Application must validate this manually ✗
 *
 *
 * ───────────────────────────────────────────────────────
 * CONCEPT 8 — Performance Characteristics
 * ───────────────────────────────────────────────────────
 *
 *   Neither is universally faster. It depends on the query.
 *
 *
 *   VISUAL — Performance trade-offs:
 *
 *   Scenario 1: Fetch user + all their orders + product names
 *
 *   Relational:
 *   One SQL query with 2 JOINs → Query Optimizer uses indexes
 *   → returns in ~5ms ✓
 *
 *   Non-Relational (embedded):
 *   One document read → no JOIN at all → returns in ~2ms ✓✓
 *   (faster because everything is co-located on disk)
 *
 *   Non-Relational (referenced):
 *   Read order doc → read user doc → read product docs
 *   → 3 separate round trips → ~15ms ✗
 *
 *
 *   Scenario 2: "Count orders per city for all users, last 30 days"
 *
 *   Relational:
 *   SELECT city, COUNT(*) FROM orders o
 *   JOIN users u ON o.user_id = u.id
 *   WHERE o.created_at > NOW() - INTERVAL '30 days'
 *   GROUP BY city
 *   → Query Optimizer + indexes → ~50ms ✓
 *
 *   Non-Relational:
 *   Aggregation pipeline with $lookup + $group
 *   → must scan many documents → ~200ms ✗
 *   (NoSQL not designed for ad-hoc cross-collection analytics)
 *
 *
 *   Scenario 3: Write 1 million records per second
 *
 *   Relational (single node):
 *   → bottlenecked at ~50,000 writes/second ✗
 *
 *   Non-Relational (Cassandra, 20 nodes):
 *   → 1,000,000 writes/second easily ✓
 *   (write-optimised + distributed by design)
 *
 *
 * ───────────────────────────────────────────────────────
 * CONCEPT 9 — Common Misconceptions
 * ───────────────────────────────────────────────────────
 *
 *   Misconception 1 — "NoSQL is always faster":
 *   - For simple document reads: sometimes yes.
 *   - For complex queries with multiple relationships: no.
 *   - For analytics on relational data: no.
 *   - For raw write throughput at massive scale: yes.
 *
 *   Misconception 2 — "SQL cannot scale":
 *   - PostgreSQL powers Instagram, Shopify, GitHub.
 *   - With read replicas and proper indexing, SQL handles
 *     hundreds of thousands of queries per second.
 *   - Horizontal sharding with Citus (PostgreSQL extension)
 *     scales to billions of rows.
 *
 *   Misconception 3 — "NoSQL means no schema":
 *   - NoSQL means no fixed schema enforced by the database.
 *   - In practice, every application imposes a schema —
 *     it just lives in application code, not the database.
 *   - Undisciplined schema = data chaos at scale.
 *
 *   Misconception 4 — "Use NoSQL for new projects":
 *   - MongoDB was recommended for everything around 2012–2016.
 *   - Many teams migrated back to PostgreSQL after hitting
 *     complex query, JOIN, and data consistency problems.
 *   - Today: PostgreSQL is the default safe choice.
 *     Use NoSQL when you have a specific reason.
 *
 *
 * ───────────────────────────────────────────────────────
 * CONCEPT 10 — Head-to-Head Comparison
 * ───────────────────────────────────────────────────────
 *
 *     +----------------------+------------------+--------------------+
 *     |  Dimension           |  Relational      |  Non-Relational    |
 *     +----------------------+------------------+--------------------+
 *     |  Data structure      |  Tables + rows   |  Docs/KV/Graph/Col |
 *     |  Schema              |  Fixed, upfront  |  Flexible, runtime |
 *     |  Relationships       |  First-class JOINs| Embed or reference|
 *     |  Scaling             |  Vertical        |  Horizontal        |
 *     |  Consistency         |  Strong (ACID)   |  Eventual (BASE)   |
 *     |  Query language      |  Standard SQL    |  DB-specific API   |
 *     |  Data integrity      |  DB enforced     |  App enforced      |
 *     |  Complex queries     |  Excellent       |  Limited           |
 *     |  Write throughput    |  Moderate        |  Very high         |
 *     |  Schema migrations   |  Painful at scale|  Effortless        |
 *     |  Learning curve      |  Low (SQL known) |  Per-DB learning   |
 *     |  ACID transactions   |  Full support    |  Limited / none    |
 *     |  Best for            |  Structured,     |  Scale, flexible   |
 *     |                      |  financial, ERP  |  schema, speed     |
 *     +----------------------+------------------+--------------------+
 *
 *
 *   Decision visual:
 *
 *    Is your data highly structured and relational?
 *         │
 *    ┌────┴────┐
 *    │         │
 *   YES        NO
 *    │         │
 *    ▼         ▼
 *    Do you need   Is your schema changing rapidly?
 *    strong ACID?       │
 *         │        ┌────┴────┐
 *    ┌────┴────┐   │         │
 *    │         │  YES        NO
 *   YES        NO  │         │
 *    │         │   ▼         ▼
 *    ▼         ▼  NoSQL   Do you need massive
 *  RDBMS     Consider  write throughput?
 *  (Postgres) NoSQL for      │
 *             flexibility ┌──┴──┐
 *                         │     │
 *                        YES    NO
 *                         │     │
 *                         ▼     ▼
 *                       NoSQL  RDBMS
 *                    (Cassandra) (Postgres
 *                               is fine)
 */

In [ ]:
/**
 * ============================================================
 * Choosing the Right Database
 * ============================================================
 *
 *   There is no universally "best" database.
 *   Every database is a trade-off between different properties:
 *   speed, consistency, flexibility, scalability, and cost.
 *
 *   The right database depends entirely on:
 *   - What kind of data you are storing
 *   - How you will read and write it
 *   - How much of it there will be
 *   - How fast it needs to be
 *   - What guarantees you need
 *
 *
 * ───────────────────────────────────────────────────────
 * FRAMEWORK — Ask These Questions First
 * ───────────────────────────────────────────────────────
 *
 *   Before picking a database, answer these questions:
 *
 *     +----+--------------------------------------------------+
 *     |  1 | What is the shape of your data?                 |
 *     |    | Structured tables? Flexible documents?          |
 *     |    | Relationships? Graph-like connections?          |
 *     +----+--------------------------------------------------+
 *     |  2 | What is your read/write pattern?                |
 *     |    | Read-heavy? Write-heavy? Both?                  |
 *     |    | Point lookups? Range queries? Analytics?        |
 *     +----+--------------------------------------------------+
 *     |  3 | How much data will you have?                    |
 *     |    | MBs? GBs? TBs? Petabytes?                      |
 *     +----+--------------------------------------------------+
 *     |  4 | Do you need ACID transactions?                  |
 *     |    | Money? Inventory? Or eventual consistency ok?   |
 *     +----+--------------------------------------------------+
 *     |  5 | How will it scale?                              |
 *     |    | Vertical (bigger machine)?                      |
 *     |    | Horizontal (more machines)?                     |
 *     +----+--------------------------------------------------+
 *     |  6 | What is your latency requirement?               |
 *     |    | Sub-millisecond? < 10ms? < 1 second?            |
 *     +----+--------------------------------------------------+
 *     |  7 | Does your schema change often?                  |
 *     |    | Fixed forever? Evolves frequently?              |
 *     +----+--------------------------------------------------+
 *
 *
 * ───────────────────────────────────────────────────────
 * CONCEPT 1 — Data Shape Drives the Choice
 * ───────────────────────────────────────────────────────
 *
 *   The single biggest factor in choosing a database is
 *   the shape of your data — how it is structured and
 *   how different pieces relate to each other.
 *
 *
 *   Shape 1 — Structured, relational data:
 *   Use: PostgreSQL, MySQL
 *
 *   - Data naturally fits into tables with rows and columns.
 *   - Entities have clear relationships (user has many orders).
 *   - Schema is well-defined and stable.
 *   - You need to query across multiple related entities (JOINs).
 *
 *    ┌──────────┐    ┌──────────┐    ┌──────────┐
 *    │  users   │───►│  orders  │───►│  items   │
 *    └──────────┘    └──────────┘    └──────────┘
 *    Clear relationships, stable schema → RDBMS ✓
 *
 *
 *   Shape 2 — Semi-structured, flexible data:
 *   Use: MongoDB, CouchDB
 *
 *   - Data has varying fields across records.
 *   - Schema evolves frequently (new fields added often).
 *   - Data is naturally hierarchical or nested.
 *   - No complex cross-entity JOINs needed most of the time.
 *
 *    User A: { name, age, courses: [...] }
 *    User B: { name, age, linkedin, github, skills: [...] }
 *    User C: { name, company, employees: 500, type: "org" }
 *    Varying shape → Document Store ✓
 *
 *
 *   Shape 3 — Simple key-based lookup:
 *   Use: Redis, DynamoDB
 *
 *   - You know the exact key to fetch — no searching needed.
 *   - Speed matters more than complex querying.
 *   - Data is ephemeral (sessions, cache) or simple (counters).
 *
 *    "session:abc123" → user session data
 *    "cache:homepage" → cached HTML
 *    "rate:user:001"  → API call count
 *    Simple key lookup → Key-Value ✓
 *
 *
 *   Shape 4 — Relationships ARE the data:
 *   Use: Neo4j, Amazon Neptune
 *
 *   - Connections between entities matter as much as the
 *     entities themselves.
 *   - Queries traverse relationships (friends of friends,
 *     paths between nodes).
 *
 *    (User) ──FOLLOWS──► (User) ──FOLLOWS──► (User)
 *    Deep relationships → Graph Database ✓
 *
 *
 *   Shape 5 — Time-series or analytical data:
 *   Use: Cassandra, InfluxDB, ClickHouse
 *
 *   - Data arrives in chronological order continuously.
 *   - Queries read large ranges across time (last 30 days).
 *   - Write throughput is very high (millions of events/sec).
 *
 *    { timestamp: 1700000001, sensor: "A", temp: 28.5 }
 *    { timestamp: 1700000002, sensor: "A", temp: 28.6 }
 *    High-volume time-ordered data → Column/Time-series ✓
 *
 *
 * ───────────────────────────────────────────────────────
 * CONCEPT 2 — Read vs Write Pattern
 * ───────────────────────────────────────────────────────
 *
 *   Different databases are optimised for different
 *   read/write ratios and access patterns.
 *
 *
 *   Read-Heavy workloads:
 *   - Most requests are reads, few are writes.
 *   - e.g., news website, product catalogue, content platform.
 *   - Use a database with good read performance and caching.
 *   - Add a read replica or a caching layer (Redis) in front.
 *
 *    Write once → Read millions of times
 *    → PostgreSQL with read replicas + Redis cache ✓
 *
 *
 *   Write-Heavy workloads:
 *   - Constant stream of incoming data — logs, events, metrics.
 *   - e.g., IoT sensors, clickstream, payment events.
 *   - Use a database optimised for fast writes.
 *   - LSM-tree based databases (Cassandra, RocksDB) shine here.
 *
 *    Millions of writes per second
 *    → Cassandra / DynamoDB ✓
 *
 *
 *   Point lookups vs Range queries:
 *
 *    Point lookup: give me user where id = 5
 *    → Key-Value store or indexed RDBMS column ✓
 *
 *    Range query: give me all orders between Jan 1 and Jan 31
 *    → RDBMS with B-tree index or time-series DB ✓
 *
 *    Analytical query: total revenue grouped by city last 90 days
 *    → OLAP database (ClickHouse, Redshift) ✓
 *
 *
 *   VISUAL — Read/Write pattern → database choice:
 *
 *    ┌────────────────────────────────────────────────────┐
 *    │  Read/Write Pattern         Best Choice            │
 *    ├────────────────────────────────────────────────────┤
 *    │  Mostly reads, stable data  PostgreSQL + Redis     │
 *    │  Mostly writes, time-series Cassandra / InfluxDB   │
 *    │  Equal reads and writes     PostgreSQL / MongoDB   │
 *    │  Instant key lookups        Redis / DynamoDB       │
 *    │  Complex analytics          ClickHouse / Redshift  │
 *    └────────────────────────────────────────────────────┘
 *
 *
 * ───────────────────────────────────────────────────────
 * CONCEPT 3 — Scale Requirements
 * ───────────────────────────────────────────────────────
 *
 *   How you plan to scale is a critical factor.
 *   Different databases scale in fundamentally different ways.
 *
 *
 *   Vertical Scaling (Scale Up):
 *   - Buy a bigger, more powerful single machine.
 *   - More CPU, more RAM, faster disk.
 *   - Simple — no code changes needed.
 *   - Has a hard limit — the biggest machine AWS sells.
 *   - Expensive at the top end.
 *   - Single point of failure.
 *
 *    ┌──────────┐       ┌──────────────┐
 *    │  4 CPU   │  ───► │   32 CPU     │
 *    │  16 GB   │       │   256 GB     │
 *    └──────────┘       └──────────────┘
 *    Good for: PostgreSQL, MySQL (traditional RDBMS)
 *
 *
 *   Horizontal Scaling (Scale Out):
 *   - Add more machines, distribute data across them.
 *   - No upper limit — just keep adding nodes.
 *   - Much cheaper at scale.
 *   - Requires the database to support sharding/distribution.
 *
 *    ┌──────┐         ┌──────┐ ┌──────┐ ┌──────┐
 *    │ Node │  ─────► │ Node │ │ Node │ │ Node │
 *    └──────┘         └──────┘ └──────┘ └──────┘
 *    Good for: MongoDB, Cassandra, DynamoDB
 *
 *
 *   Sharding — how horizontal scaling works:
 *   - Sharding splits your data across multiple nodes.
 *   - Each node (shard) holds a portion of the data.
 *   - e.g., users A-M go to Shard 1, N-Z go to Shard 2.
 *
 *    ┌──────────────────────────────────────────────────┐
 *    │  10 million users distributed across 4 shards    │
 *    │                                                  │
 *    │  Shard 1: users A-D   (2.5M users)              │
 *    │  Shard 2: users E-L   (2.5M users)              │
 *    │  Shard 3: users M-R   (2.5M users)              │
 *    │  Shard 4: users S-Z   (2.5M users)              │
 *    │                                                  │
 *    │  Query user "Hitesh" → goes to Shard 1 directly │
 *    └──────────────────────────────────────────────────┘
 *
 *
 * ───────────────────────────────────────────────────────
 * CONCEPT 4 — Consistency Requirements
 * ───────────────────────────────────────────────────────
 *
 *   Ask: what happens if two users read slightly different
 *   values for the same data?
 *   - Is that acceptable for your use case?
 *   - Or does it cause serious problems?
 *
 *
 *   Strong Consistency needed:
 *   - Every read must return the most recent write.
 *   - No stale data ever.
 *   - Required for: banking, payments, inventory, auth.
 *   → Use: PostgreSQL, MySQL, MongoDB (CP mode)
 *
 *
 *   Eventual Consistency acceptable:
 *   - Data might be slightly stale for a short window.
 *   - All nodes converge to the same value eventually.
 *   - Acceptable for: social feeds, view counts, carts.
 *   → Use: DynamoDB, Cassandra, CouchDB
 *
 *
 *   VISUAL — Consistency vs Speed trade-off:
 *
 *    Strong Consistency ◄────────────────► Eventual Consistency
 *         │                                        │
 *    Always correct                          Always fast
 *    May be slower                           May be stale
 *    PostgreSQL                              DynamoDB
 *    MongoDB (CP)                            Cassandra
 *
 *
 * ───────────────────────────────────────────────────────
 * CONCEPT 5 — Schema Flexibility
 * ───────────────────────────────────────────────────────
 *
 *   Fixed schema (RDBMS):
 *   - You define the schema upfront.
 *   - Every row must conform to the schema.
 *   - Adding a new column requires a schema migration
 *     (ALTER TABLE) — can be slow on large tables.
 *   - Great when your data model is stable and well understood.
 *
 *    ALTER TABLE users ADD COLUMN linkedin VARCHAR(200);
 *    ← runs on every existing row — 10M rows = slow ✗
 *
 *
 *   Flexible schema (NoSQL):
 *   - Each document can have different fields.
 *   - No migration needed — just start storing new fields.
 *   - Great for rapidly evolving products or prototypes.
 *   - Risk: inconsistent data if not managed carefully.
 *
 *    // Just start storing linkedin — no migration needed ✓
 *    { name: "Hitesh", linkedin: "h/hitesh" }
 *
 *
 *   Schema flexibility decision:
 *
 *    Do you know your full data model upfront?
 *         │
 *    ┌────┴────┐
 *    │         │
 *   YES        NO
 *    │         │
 *    ▼         ▼
 *   RDBMS     NoSQL Document
 *   (fixed    (flexible
 *   schema)   schema)
 *
 *
 * ───────────────────────────────────────────────────────
 * CONCEPT 6 — The Polyglot Persistence Pattern
 * ───────────────────────────────────────────────────────
 *
 *   Real-world systems almost never use just one database.
 *   Different parts of the system have different needs.
 *   Using the best database for each use case is called
 *   Polyglot Persistence.
 *
 *
 *   Example — ChaiCode LMS platform using multiple databases:
 *
 *    ┌─────────────────────────────────────────────────────────────┐
 *    │                   ChaiCode Backend                           │
 *    └─────┬──────────┬──────────┬──────────┬──────────┬──────────┘
 *          │          │          │          │          │
 *          ▼          ▼          ▼          ▼          ▼
 *    ┌──────────┐ ┌────────┐ ┌───────┐ ┌────────┐ ┌──────────┐
 *    │PostgreSQL│ │MongoDB │ │ Redis │ │Elastic-│ │Cassandra │
 *    │          │ │        │ │       │ │ search │ │          │
 *    │ Users    │ │ Course │ │Session│ │ Course │ │ Video    │
 *    │ Payments │ │ Content│ │ Cache │ │ Search │ │ Watch    │
 *    │ Orders   │ │ Quizzes│ │ Rate  │ │        │ │ Events   │
 *    │          │ │        │ │ Limit │ │        │ │          │
 *    └──────────┘ └────────┘ └───────┘ └────────┘ └──────────┘
 *    ACID needed  Flexible    Speed    Full-text  High-write
 *    for payments schema      needed   search     time-series
 *
 *
 *   Each database is chosen for what it does best:
 *
 *     +------------------+------------+----------------------------+
 *     |  Data            |  Database  |  Why                       |
 *     +------------------+------------+----------------------------+
 *     |  Users, payments |  PostgreSQL|  ACID, relational, stable  |
 *     |  Course content  |  MongoDB   |  Flexible, nested content  |
 *     |  Sessions, cache |  Redis     |  Sub-ms, ephemeral data    |
 *     |  Course search   |  Elastic   |  Full-text search engine   |
 *     |  Watch events    |  Cassandra |  High write, time-series   |
 *     +------------------+------------+----------------------------+
 *
 *
 * ───────────────────────────────────────────────────────
 * DECISION GUIDE — Matching Use Case to Database
 * ───────────────────────────────────────────────────────
 *
 *     +------------------------------+-----------------------------+
 *     |  Use Case                    |  Best Database Choice       |
 *     +------------------------------+-----------------------------+
 *     |  User accounts, auth         |  PostgreSQL / MySQL         |
 *     |  Financial transactions      |  PostgreSQL / MySQL         |
 *     |  E-commerce orders           |  PostgreSQL / MySQL         |
 *     |  CMS, blog posts             |  MongoDB / PostgreSQL       |
 *     |  Product catalogue           |  MongoDB / PostgreSQL       |
 *     |  User sessions               |  Redis                      |
 *     |  API rate limiting           |  Redis                      |
 *     |  Leaderboards / rankings     |  Redis (sorted sets)        |
 *     |  Caching layer               |  Redis                      |
 *     |  Full-text search            |  Elasticsearch              |
 *     |  Social network graph        |  Neo4j / Neptune            |
 *     |  Fraud detection             |  Neo4j                      |
 *     |  IoT sensor data             |  InfluxDB / Cassandra       |
 *     |  Application logs            |  Cassandra / Elasticsearch  |
 *     |  Analytics / reporting       |  ClickHouse / Redshift      |
 *     |  Real-time messaging         |  Redis / Cassandra          |
 *     |  Shopping cart               |  Redis / DynamoDB           |
 *     |  Config / feature flags      |  Redis / DynamoDB           |
 *     +------------------------------+-----------------------------+
 *
 *
 * ───────────────────────────────────────────────────────
 * COMPLETE DECISION FLOWCHART
 * ───────────────────────────────────────────────────────
 *
 *    What kind of data are you storing?
 *         │
 *    ┌────┴───────────────────────────────────────┐
 *    │                                            │
 *    Relationships matter                    Simple key lookup
 *    (users, orders, tables)                 (cache, session)
 *    │                                            │
 *    ▼                                            ▼
 *    Are the relationships                    Redis / DynamoDB
 *    the core of the data?
 *    (social graph, fraud)
 *    │
 *    ┌────┴────┐
 *    │         │
 *   YES        NO
 *    │         │
 *    ▼         │
 *  Graph DB    │
 *  (Neo4j)     Is your schema fixed or flexible?
 *              │
 *         ┌────┴────┐
 *         │         │
 *       FIXED    FLEXIBLE
 *         │         │
 *         │         ▼
 *         │    Document Store (MongoDB)
 *         │
 *         Do you need strong ACID?
 *         │
 *    ┌────┴────┐
 *    │         │
 *   YES        NO
 *    │         │
 *    ▼         ▼
 * PostgreSQL  Is it time-series or analytics?
 * MySQL       │
 *        ┌────┴────┐
 *        │         │
 *       YES        NO
 *        │         │
 *        ▼         ▼
 *   Cassandra   DynamoDB
 *   InfluxDB    MongoDB
 *
 *
 * ───────────────────────────────────────────────────────
 * REAL-WORLD EXAMPLES — What Big Companies Use
 * ───────────────────────────────────────────────────────
 *
 *     +------------------+--------------------------------------------+
 *     |  Company         |  Database choices and why                  |
 *     +------------------+--------------------------------------------+
 *     |  Instagram       |  PostgreSQL (posts, users)                 |
 *     |                  |  Cassandra (activity feeds)                |
 *     |                  |  Redis (sessions, cache)                   |
 *     +------------------+--------------------------------------------+
 *     |  Uber            |  MySQL (trips, payments)                   |
 *     |                  |  Schemaless (MySQL + NoSQL layer on top)   |
 *     |                  |  Redis (real-time driver location)         |
 *     +------------------+--------------------------------------------+
 *     |  Netflix         |  MySQL (billing, user data)                |
 *     |                  |  Cassandra (viewing history)               |
 *     |                  |  DynamoDB (session data)                   |
 *     |                  |  Elasticsearch (search)                    |
 *     +------------------+--------------------------------------------+
 *     |  Twitter/X       |  MySQL (tweets, users)                     |
 *     |                  |  Redis (timeline cache)                    |
 *     |                  |  Manhattan (custom key-value, DynamoDB     |
 *     |                  |  inspired)                                 |
 *     +------------------+--------------------------------------------+
 *     |  MongoDB itself  |  Uses PostgreSQL for billing ✓             |
 *     |                  |  (right tool for right job)                |
 *     +------------------+--------------------------------------------+
 *
 *
 *   Summary — database selection cheat sheet:
 *
 *     +------------------+------------------------------------------+
 *     |  Database        |  Choose when...                          |
 *     +------------------+------------------------------------------+
 *     |  PostgreSQL      |  Relational data, ACID, complex queries  |
 *     +------------------+------------------------------------------+
 *     |  MySQL           |  Relational data, high read throughput   |
 *     +------------------+------------------------------------------+
 *     |  MongoDB         |  Flexible schema, nested documents,      |
 *     |                  |  evolving data model                     |
 *     +------------------+------------------------------------------+
 *     |  Redis           |  Caching, sessions, rate limiting,       |
 *     |                  |  leaderboards — speed is everything      |
 *     +------------------+------------------------------------------+
 *     |  DynamoDB        |  Massive scale, key-value or document,   |
 *     |                  |  AWS-native, serverless                  |
 *     +------------------+------------------------------------------+
 *     |  Cassandra       |  Huge write volume, time-series,         |
 *     |                  |  no single point of failure              |
 *     +------------------+------------------------------------------+
 *     |  Elasticsearch   |  Full-text search, log analysis          |
 *     +------------------+------------------------------------------+
 *     |  Neo4j           |  Graph data, social networks, fraud      |
 *     +------------------+------------------------------------------+
 *     |  InfluxDB        |  IoT, metrics, time-series data          |
 *     +------------------+------------------------------------------+
 *     |  ClickHouse      |  Real-time analytics on huge datasets    |
 *     +------------------+------------------------------------------+
 */


In [ ]:
/**
 * ============================================================
 * The Paradigm Shift Toward Real-Time Databases
 * ============================================================
 *
 *   Traditional databases work on a request-response model:
 *   - Your app asks a question → database answers → done.
 *   - The app has to keep asking to see if anything changed.
 *   - This worked fine for 20 years — but the world changed.
 *
 *   Modern applications need to REACT to data changes instantly:
 *   - WhatsApp message delivered → recipient sees it immediately
 *   - Swiggy order status changes → customer sees it instantly
 *   - Stock price changes → trader's screen updates in real time
 *   - Uber driver moves → passenger sees the car move on map
 *
 *   This shift from "ask and receive" to "listen and react"
 *   is the paradigm shift toward real-time databases.
 *
 *
 * ───────────────────────────────────────────────────────
 * CONCEPT 1 — The Old Way: Polling
 * ───────────────────────────────────────────────────────
 *
 *   - In the traditional model, the client repeatedly asks
 *     the server "has anything changed?"
 *   - This is called Polling.
 *   - The client does not know when data changes — it just
 *     keeps checking at fixed intervals.
 *
 *
 *   VISUAL — Polling (the old way):
 *
 *    [Client App]                         [Database]
 *         │
 *         │─── "Any new messages?" ───────►│
 *         │◄─── "No" ─────────────────────│
 *         │  (wait 2 seconds)              │
 *         │─── "Any new messages?" ───────►│
 *         │◄─── "No" ─────────────────────│
 *         │  (wait 2 seconds)              │
 *         │─── "Any new messages?" ───────►│  ← message arrives here
 *         │◄─── "Yes! Here it is" ────────│
 *         │  (wait 2 seconds)              │
 *         │─── "Any new messages?" ───────►│
 *         │◄─── "No" ─────────────────────│
 *
 *
 *   Problems with Polling:
 *
 *     +------------------------------+----------------------------------+
 *     |  Problem                     |  Impact                          |
 *     +------------------------------+----------------------------------+
 *     |  Wasted requests             |  99% of polls return "No" —      |
 *     |                              |  all that compute is wasted      |
 *     +------------------------------+----------------------------------+
 *     |  Latency                     |  Message arrives but user sees   |
 *     |                              |  it only on the next poll (up    |
 *     |                              |  to 2 seconds late)              |
 *     +------------------------------+----------------------------------+
 *     |  Server load                 |  1000 users polling every 2s =   |
 *     |                              |  500 req/sec of wasted traffic   |
 *     +------------------------------+----------------------------------+
 *     |  Battery drain               |  Mobile apps drain battery fast  |
 *     |                              |  with constant network requests  |
 *     +------------------------------+----------------------------------+
 *
 *
 *   VISUAL — Polling cost with 1000 users:
 *
 *    1000 users × 1 poll every 2 seconds
 *    = 500 requests per second to the database
 *
 *    Of those 500 requests:
 *    ┌─────────────────────────────────────────┐
 *    │  495 requests → "Nothing changed" ✗     │
 *    │    5 requests → "Here is new data" ✓    │
 *    └─────────────────────────────────────────┘
 *    99% of server work is wasted ✗
 *
 *
 * ───────────────────────────────────────────────────────
 * CONCEPT 2 — The New Way: Push / Subscriptions
 * ───────────────────────────────────────────────────────
 *
 *   - Instead of the client asking repeatedly, the client
 *     subscribes once and the database/server pushes
 *     updates to the client the moment data changes.
 *   - The client is passive — it listens instead of asking.
 *   - This is called the Push Model or Reactive Model.
 *
 *
 *   VISUAL — Push model (the new way):
 *
 *    [Client App]                         [Database]
 *         │
 *         │─── "Subscribe to messages" ──►│
 *         │◄─── "OK, I'll notify you" ────│
 *         │                               │
 *         │  (client does nothing — waits)│
 *         │                               │  ← message arrives
 *         │◄─── "New message!" ───────────│  ← database pushes instantly
 *         │                               │
 *         │  (client does nothing — waits)│
 *         │                               │  ← another message
 *         │◄─── "New message!" ───────────│  ← pushed instantly
 *
 *    Zero wasted requests ✓
 *    Zero latency ✓  (pushed the moment it changes)
 *    Connection stays open — no repeated requests ✓
 *
 *
 *   Polling vs Push comparison:
 *
 *     +------------------+--------------------+--------------------+
 *     |  Feature         |  Polling           |  Push/Subscribe    |
 *     +------------------+--------------------+--------------------+
 *     |  Who initiates   |  Client repeatedly |  Server/DB pushes  |
 *     |  communication   |  asks              |  when data changes |
 *     +------------------+--------------------+--------------------+
 *     |  Latency         |  Up to poll interval| Milliseconds      |
 *     +------------------+--------------------+--------------------+
 *     |  Wasted requests |  Huge              |  Zero              |
 *     +------------------+--------------------+--------------------+
 *     |  Server load     |  High              |  Low               |
 *     +------------------+--------------------+--------------------+
 *     |  Complexity      |  Simple            |  More complex      |
 *     +------------------+--------------------+--------------------+
 *     |  Battery usage   |  High (mobile)     |  Low               |
 *     +------------------+--------------------+--------------------+
 *
 *
 * ───────────────────────────────────────────────────────
 * CONCEPT 3 — Technologies That Enable Real-Time
 * ───────────────────────────────────────────────────────
 *
 *   Three main technologies power real-time communication:
 *
 *
 *   Technology 1 — WebSockets:
 *   - A persistent, two-way connection between client and server.
 *   - Once the WebSocket connection is open, both sides can
 *     send messages to each other at any time.
 *   - No repeated HTTP requests — one connection stays open.
 *   - Used by: WhatsApp Web, Slack, online games, live chat.
 *
 *
 *   VISUAL — HTTP vs WebSocket:
 *
 *    HTTP (open → request → response → close):
 *    Client ──open──► Server ──response──► Client ──close──►
 *           ──open──► Server ──response──► Client ──close──►
 *           ──open──► Server ──response──► Client ──close──►
 *    New connection every time — expensive ✗
 *
 *    WebSocket (open once → keep talking):
 *    Client ──── OPEN CONNECTION ────► Server
 *              ◄──── message ─────────
 *              ──── message ──────────►
 *              ◄──── message ─────────
 *              ──── message ──────────►
 *    Client ──── CLOSE CONNECTION ───► Server
 *    One connection, bidirectional, as long as needed ✓
 *
 *
 *   Technology 2 — Server-Sent Events (SSE):
 *   - One-way push from server to client over HTTP.
 *   - Client subscribes once — server streams updates.
 *   - Simpler than WebSocket — good when client does not
 *     need to send data back (notifications, live feeds).
 *   - Used by: news tickers, live dashboards, notifications.
 *
 *
 *   VISUAL — SSE (one-way stream):
 *
 *    Client ────── GET /subscribe ────────► Server
 *           ◄────── event: update ──────────
 *           ◄────── event: update ──────────
 *           ◄────── event: update ──────────
 *    (client only receives — cannot send back) ✓
 *
 *
 *   Technology 3 — Long Polling:
 *   - A middle ground between regular polling and WebSockets.
 *   - Client sends a request — server holds the connection
 *     open (does not respond immediately).
 *   - When data is available, server responds and closes.
 *   - Client immediately sends the next request.
 *   - Used by: older chat systems before WebSockets.
 *
 *
 *   VISUAL — Long Polling:
 *
 *    Client ─── request ───────────────────► Server
 *           ← (server holds, waits for data) →
 *           ◄── response (when data available)─
 *    Client ─── request immediately ──────── ► Server
 *           ← (server holds again...)
 *
 *    Better than polling but still reconnects constantly ✗
 *    WebSocket is the modern replacement ✓
 *
 *
 * ───────────────────────────────────────────────────────
 * CONCEPT 4 — Real-Time Databases
 * ───────────────────────────────────────────────────────
 *
 *   - A real-time database builds the push mechanism directly
 *     into the database itself.
 *   - Instead of just storing and querying data, it also
 *     streams changes to all connected clients instantly.
 *   - You subscribe to a piece of data. When that data
 *     changes in the database, every subscriber is notified
 *     automatically — without asking.
 *
 *
 *   Traditional DB vs Real-Time DB:
 *
 *    Traditional (pull):
 *    ┌──────────┐  query   ┌────────────┐
 *    │  Client  │ ───────► │  Database  │
 *    │          │ ◄─────── │            │
 *    └──────────┘ response └────────────┘
 *    Client always initiates. DB passively answers.
 *
 *    Real-Time DB (push):
 *    ┌──────────┐ subscribe┌────────────┐
 *    │  Client  │ ───────► │  Real-Time │
 *    │          │ ◄─────── │  Database  │
 *    │          │  push    │            │
 *    │          │ ◄─────── │ (on change)│
 *    │          │  push    │            │
 *    │          │ ◄─────── │ (on change)│
 *    └──────────┘          └────────────┘
 *    DB actively pushes every time data changes. ✓
 *
 *
 * ───────────────────────────────────────────────────────
 * CONCEPT 5 — Firebase Realtime Database
 * ───────────────────────────────────────────────────────
 *
 *   - Firebase Realtime Database (by Google) was one of the
 *     first databases to make real-time sync mainstream.
 *   - Data is stored as one large JSON tree.
 *   - All clients connected to the same path get updates
 *     the instant any client writes to that path.
 *   - Built-in offline support — changes sync when reconnected.
 *
 *
 *   VISUAL — Firebase Realtime sync:
 *
 *    Database state:
 *    /chats/room-01/messages/
 *    └── msg001: { text: "Hello", user: "Hitesh" }
 *
 *    Client A (Hitesh's phone) subscribes to /chats/room-01/
 *    Client B (Piyush's phone) subscribes to /chats/room-01/
 *    Client C (Aman's phone)   subscribes to /chats/room-01/
 *
 *    Hitesh writes a new message:
 *    /chats/room-01/messages/msg002 = { text: "Hey!", user:"Hitesh" }
 *         │
 *         ▼
 *    Firebase detects change
 *         │
 *         ├──► Client B (Piyush) receives: "Hey!" instantly ✓
 *         └──► Client C (Aman)   receives: "Hey!" instantly ✓
 *
 *    All subscribers see the update in < 100ms ✓
 *    No polling, no manual refresh needed ✓
 *
 *
 *   Firebase code example (JavaScript):
 *
 *   // Subscribe — fires every time data changes
 *   const ref = firebase.database().ref('/chats/room-01/messages');
 *
 *   ref.on('value', (snapshot) => {
 *     const messages = snapshot.val();
 *     renderMessages(messages);   // update UI automatically
 *   });
 *
 *   // Write — all subscribers notified instantly
 *   ref.push({
 *     text: "Hey!",
 *     user: "Hitesh",
 *     timestamp: Date.now()
 *   });
 *
 *   The .on('value') listener fires:
 *   - Immediately when first subscribed (initial data)
 *   - Every time ANY change happens to that path ✓
 *
 *
 * ───────────────────────────────────────────────────────
 * CONCEPT 6 — Change Data Capture (CDC)
 * ───────────────────────────────────────────────────────
 *
 *   - CDC is a pattern where the database's own write log
 *     (WAL) is monitored for changes.
 *   - Every INSERT, UPDATE, DELETE that happens in the DB
 *     is captured and streamed to other systems.
 *   - This brings real-time behaviour to traditional databases
 *     like PostgreSQL and MySQL without changing the app code.
 *   - Tool: Debezium (open source CDC connector).
 *
 *
 *   VISUAL — CDC pipeline:
 *
 *    [Your App]
 *         │
 *         │  normal INSERT/UPDATE/DELETE
 *         ▼
 *    [PostgreSQL]
 *    WAL log: "user id=5 email changed to new@..."
 *         │
 *         │  CDC tool (Debezium) reads WAL
 *         ▼
 *    [Kafka / Event Stream]
 *    event: { table:"users", op:"UPDATE", id:5, email:"new@..." }
 *         │
 *         ├──► [Search Index] update Elasticsearch ✓
 *         ├──► [Cache] invalidate Redis cache for user:5 ✓
 *         ├──► [Analytics] update data warehouse ✓
 *         └──► [WebSocket Server] push to connected client ✓
 *
 *    One DB write → many systems updated in real time ✓
 *    No polling anywhere in the pipeline ✓
 *
 *
 * ───────────────────────────────────────────────────────
 * CONCEPT 7 — MongoDB Change Streams
 * ───────────────────────────────────────────────────────
 *
 *   - MongoDB has a built-in feature called Change Streams.
 *   - Your application can subscribe to changes on a
 *     collection, database, or entire cluster.
 *   - Every insert, update, or delete fires an event to
 *     all subscribers — no polling, no CDC tool needed.
 *
 *
 *   VISUAL — MongoDB Change Stream:
 *
 *    [Orders Collection]
 *    Document { _id:"o1", status:"pending" }
 *         │
 *         │  App A updates: status = "shipped"
 *         ▼
 *    MongoDB detects change → fires to all stream subscribers
 *         │
 *         ├──► App B (notification service):
 *         │    "Order shipped! Send email to customer"  ✓
 *         │
 *         └──► App C (dashboard):
 *              "Update live order count on screen"      ✓
 *
 *
 *   MongoDB Change Stream code (Node.js):
 *
 *   const collection = db.collection('orders');
 *   const changeStream = collection.watch();
 *
 *   changeStream.on('change', (event) => {
 *     console.log('Change detected:', event.operationType);
 *     // insert | update | delete | replace
 *
 *     if (event.operationType === 'update') {
 *       const updatedFields = event.updateDescription.updatedFields;
 *       if (updatedFields.status === 'shipped') {
 *         sendShippingEmail(event.documentKey._id);
 *       }
 *     }
 *   });
 *
 *
 * ───────────────────────────────────────────────────────
 * CONCEPT 8 — PostgreSQL LISTEN / NOTIFY
 * ───────────────────────────────────────────────────────
 *
 *   - PostgreSQL has a built-in pub/sub system called
 *     LISTEN and NOTIFY.
 *   - One client can NOTIFY a channel when something happens.
 *   - All clients LISTENing to that channel receive the
 *     notification instantly.
 *   - Works at the database level — no extra tools needed.
 *
 *
 *   VISUAL — PostgreSQL LISTEN/NOTIFY:
 *
 *    Client A (backend API):
 *    INSERT INTO orders (status) VALUES ('shipped');
 *    NOTIFY order_updates, 'order_id=123';
 *         │
 *         ▼
 *    PostgreSQL broadcasts to all listeners
 *         │
 *         ├──► Client B (notification worker):
 *         │    LISTEN order_updates;
 *         │    Receives: 'order_id=123'
 *         │    → sends push notification to user
 *         │
 *         └──► Client C (dashboard):
 *              LISTEN order_updates;
 *              Receives: 'order_id=123'
 *              → updates live chart on screen
 *
 *
 *   SQL commands:
 *
 *   -- Client B and C listen on a channel:
 *   LISTEN order_updates;
 *
 *   -- Client A notifies when something happens:
 *   NOTIFY order_updates, 'order_id=123';
 *
 *   -- Can also trigger from a DB trigger automatically:
 *   CREATE FUNCTION notify_order_change()
 *   RETURNS trigger AS $$
 *   BEGIN
 *     NOTIFY order_updates, NEW.id::text;
 *     RETURN NEW;
 *   END;
 *   $$ LANGUAGE plpgsql;
 *
 *   CREATE TRIGGER order_change_trigger
 *   AFTER INSERT OR UPDATE ON orders
 *   FOR EACH ROW EXECUTE FUNCTION notify_order_change();
 *
 *   Now any INSERT or UPDATE on orders → automatically
 *   notifies all listeners — zero app code change ✓
 *
 *
 * ───────────────────────────────────────────────────────
 * CONCEPT 9 — Real-World Architecture Patterns
 * ───────────────────────────────────────────────────────
 *
 *   Pattern 1 — Chat Application (WhatsApp-style):
 *
 *    [User A sends message]
 *         │  POST /message
 *         ▼
 *    [API Server]
 *    Saves message to DB
 *         │
 *         │  WebSocket push
 *         ▼
 *    [WebSocket Server]
 *    Finds active connection for User B
 *         │
 *         ▼
 *    [User B's browser / app]
 *    Message appears instantly ✓  (< 100ms)
 *
 *
 *   Pattern 2 — Live Order Tracking (Swiggy-style):
 *
 *    [Delivery partner app updates location]
 *         │  every 3 seconds
 *         ▼
 *    [Database: driver location = {lat, lng}]
 *    Change Stream / CDC detects update
 *         │
 *         │  WebSocket push to customer
 *         ▼
 *    [Customer's app]
 *    Driver icon moves on map in real time ✓
 *
 *
 *   Pattern 3 — Live Dashboard (Analytics):
 *
 *    [Orders coming in all day]
 *         │
 *         ▼
 *    [Database: new rows inserted]
 *    CDC / Change Stream captures each insert
 *         │
 *         ▼
 *    [Stream Processor: Kafka + Flink]
 *    Aggregates in real time: revenue, order count
 *         │
 *         ▼
 *    [Dashboard via SSE or WebSocket]
 *    Numbers update live on screen ✓
 *    No page refresh needed ✓
 *
 *
 *   Pattern 4 — Real-Time Collaboration (Google Docs-style):
 *
 *    [User A types a character]
 *         │  WebSocket send
 *         ▼
 *    [Server: saves change, broadcasts to all editors]
 *         │
 *         ├──► User B sees the character appear ✓
 *         └──► User C sees the character appear ✓
 *    All editors see changes < 50ms apart ✓
 *
 *
 * ───────────────────────────────────────────────────────
 * CONCEPT 10 — The Full Paradigm Shift Summarised
 * ───────────────────────────────────────────────────────
 *
 *   VISUAL — How the database's role has evolved:
 *
 *    1990s — 2010s (Traditional):
 *    ┌──────────────────────────────────────────────────┐
 *    │  Database = passive storage                       │
 *    │  App asks → DB answers → done                    │
 *    │  Client always pulls data                        │
 *    └──────────────────────────────────────────────────┘
 *
 *    2010s — present (Real-Time):
 *    ┌──────────────────────────────────────────────────┐
 *    │  Database = active participant                    │
 *    │  DB detects change → notifies all subscribers    │
 *    │  Client listens for pushed data                  │
 *    └──────────────────────────────────────────────────┘
 *
 *
 *   The shift in one line:
 *   FROM: "App asks the database what changed"
 *   TO:   "Database tells the app what changed"
 *
 *
 *   Technologies enabling the shift:
 *
 *     +------------------------+------------------------------------+
 *     |  Technology            |  Role                             |
 *     +------------------------+------------------------------------+
 *     |  WebSockets            |  Persistent two-way connection    |
 *     |                        |  between client and server        |
 *     +------------------------+------------------------------------+
 *     |  Server-Sent Events    |  One-way server push over HTTP    |
 *     +------------------------+------------------------------------+
 *     |  Firebase Realtime DB  |  Built-in sync to all clients     |
 *     +------------------------+------------------------------------+
 *     |  MongoDB Change Streams|  Subscribe to collection changes  |
 *     +------------------------+------------------------------------+
 *     |  PostgreSQL            |  Database-level pub/sub           |
 *     |  LISTEN/NOTIFY         |  built into the query language    |
 *     +------------------------+------------------------------------+
 *     |  CDC (Debezium)        |  Stream WAL changes to any system |
 *     +------------------------+------------------------------------+
 *
 *
 *   When to use real-time:
 *
 *     +------------------------------+-------------------------------+
 *     |  Use real-time for           |  Polling is fine for          |
 *     +------------------------------+-------------------------------+
 *     |  Chat / messaging            |  Generating reports           |
 *     |  Live order / delivery track |  Batch data processing        |
 *     |  Collaborative editing       |  Daily email digests          |
 *     |  Live dashboards             |  User profile updates         |
 *     |  Stock / sports scores       |  Content management           |
 *     |  Multiplayer games           |  Search indexing              |
 *     |  IoT sensor readings         |  Billing / invoicing          |
 *     +------------------------------+-------------------------------+
 */

In [ ]:
/**
 * Constraints:
 * 1. NOT NULL: Ensures that column cannot have a null value
 * 2. UNIQUE  : Ensures that column is unique
 * 3, PRIMARY KEY: Primary Key is the attribute by which we can
 *    identify record uniquely (NOT NULL + UNIQUE)
 * 4. FOREIGN KEY: Ensures referene integrity between two tables
 *    like a bridge.
 * 5. AUTO INCREMENT: Automatically increment the value each time
 *    by 1.
 * 6. DEFAULT: Used to give default values for columns.
 * 7. CHECK  : Ensures that all values follows a specific condition
 * 8. CANDIDATE KEY: It is eligible to become primary but not opted
 *    as primary.
*/